In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon
import seaborn as sns
from datetime import datetime
import math
from matplotlib.patches import Arc
from matplotlib.patches import Ellipse
import joblib
import os
import subprocess
import sys
import lightgbm
import xgboost as xgb
import catboost
from matplotlib.colors import Normalize


try:
    import sklearn
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn"])
    import sklearn

import sklearn

def Trumedia_feature_engineering(df):
    import pandas as pd
    import numpy as np

    # -------------------------------
    # 1. Define pitch mapping
    # -------------------------------
    pitch_mapping = {
        'FA': 'Fast',
        'CU': 'Break',
        'CH': 'Slow',
        'SL': 'Break',
        'SI': 'Fast',
        'FC': 'Fast',
        'UN': None,  # filter out
        'IN': None,  # filter out
        'FF': 'Fast',
        'FS': 'Slow',
        'KN': 'Slow'
    }

    # Filter out rows where pitchType maps to None
    df = df[df['pitchType'].isin([pt for pt, group in pitch_mapping.items() if group is not None])].copy()
    # Map pitch types to a new column 'pitchgroup'
    df['pitchgroup'] = df['pitchType'].map(pitch_mapping)

    df['Autopitchtype'] =  df['Taggedpitchtype']

    # -------------------------------
    # 2. Categorize the pitchResult
    # -------------------------------
    def categorize_event(event):
        """
        Categorize the pitchResult into a standardized event label.
        Bunt events are ignored (return None).
        """
        event = event.lower()
        if "bunt" in event or "unknown" in event:
            return None
        elif "single" in event:
            return "single"
        elif "double play" in event:
            return "field_out"
        elif "double" in event:
            return "double"
        elif "triple" in event:
            return "triple"
        elif "home run" in event:
            return "home_run"
        elif "looking" in event:
            return "called_strike"
        elif "swinging" in event:
            return "swinging_strike"
        elif "hit by pitch" in event:
            return "hit_by_pitch"
        elif "walk" in event or "ball" in event:
            return "ball"
        elif "foul" in event:
            return "foul"
        elif ("line out" in event or "fly out" in event or "ground out" in event or
              "pop out" in event or "double play" in event or "reached on error" in event or
              "in play out" in event or "sac fly" in event or "fielder's choice" in event):
            return "field_out"
        elif "ball in the dirt" in event:
            return "ball"
        else:
            return "unknown"

    df['event_category'] = df['pitchResult'].apply(categorize_event)
    # Drop rows where event_category is None or unknown
    df = df[(df['event_category'].notna()) & (df['event_category'] != 'unknown')].copy()

    # -------------------------------
    # 3. Split 'count' into balls and strikes
    # -------------------------------
    df[['balls', 'strikes']] = df['count'].str.split('-', expand=True)
    df['balls'] = pd.to_numeric(df['balls'], errors='coerce')
    df['strikes'] = pd.to_numeric(df['strikes'], errors='coerce')

    # -------------------------------
    # 4. Merge with run_values
    # -------------------------------
    run_values = pd.read_csv("https://raw.githubusercontent.com/tdub29/streamlit-app-1/refs/heads/main/run_values.csv")
    run_values = run_values.rename(columns={"event": "event_category"})
    df_joined = pd.merge(
        df,
        run_values,
        on=["balls", "strikes", "event_category"],
        how="left"
    )



    # -------------------------------
    # 5. Create Binary 'win' Column
    # -------------------------------
    win_events = {'foul', 'called_strike', 'swinging_strike', 'field_out', 'strikeout'}
    df_joined['win'] = df_joined['event_category'].apply(lambda x: 1 if x in win_events else 0)

    # -------------------------------
    # 6. Adjust PX Orientation
    # -------------------------------
    df_joined['PX'] = df_joined['PX'] * -1

    # -------------------------------
    # 7. Create Binary Count Categories
    # -------------------------------
    df_joined['count_0_0'] = ((df_joined['balls'] == 0) & (df_joined['strikes'] == 0)).astype(int)
    df_joined['count_hitters'] = df_joined[['balls', 'strikes']].apply(
        lambda x: 1 if (x['balls'], x['strikes']) in [(1,0), (2,0), (3,0), (3,1)] else 0, axis=1
    )
    df_joined['count_pitchers'] = df_joined[['balls', 'strikes']].apply(
        lambda x: 1 if (x['balls'], x['strikes']) in [(0,2), (0,1), (1,2)] else 0, axis=1
    )
    df_joined['count_2k'] = ((df_joined['strikes'] == 2) & (df_joined['balls'] != 3)).astype(int)

    # -------------------------------
    # 8. Additional Binary Features
    # -------------------------------
    # 8.1 Strike
    strike_events = {"foul", "called_strike", "swinging_strike", "field_out", "strikeout",
                     "home_run", "triple", "double", "single"}
    df_joined['Strike'] = df_joined['event_category'].isin(strike_events)

    # # 8.2 Comploc
    # df_joined['Comploc'] = df_joined.apply(
    #     lambda row: -1.15 <= row['PX'] <= 1.15 and 1.1 <= row['PZ'] <= 3.9, axis=1
    # )

    # # 8.3 Inzone
    # df_joined['Inzone'] = df_joined.apply(
    #     lambda row: -0.83 <= row['PX'] <= 0.83 and 1.5 <= row['PZ'] <= 3.5, axis=1
    # )

    # 8.4 Swing
    swing_events = {"foul", "swinging_strike", "field_out", "home_run", "triple", "double", "single"}
    df_joined['Swing'] = df_joined['event_category'].isin(swing_events)

    # 8.5 Whiff
    df_joined['Whiff'] = df_joined['pitchResult'].str.lower().str.contains("swinging", na=False).astype(int)

    # -------------------------------
    # 9. Create delta_run_exp_squared
    # -------------------------------
    df_joined['delta_run_exp_squared'] = df_joined['delta_run_exp'].apply(lambda x:
        0.5 + (x - 0.5) * 0.5 if x > 0.5 else
        -0.5 + (x + 0.5) * 0.5 if x < -0.5 else
        0.2 + (x - 0.2) * 0.75 if x > 0.2 else
        -0.2 + (x + 0.2) * 0.75 if x < -0.2 else
        x * 2.5
    )

    # -------------------------------
    # 10. Convert Numeric Columns
    # -------------------------------
    numeric_cols = ["Vel", "delta_run_exp", "Extension", "HorzApprAngle", "VertApprAngle", 
                    "IndVertBrk", "HorzBrk", "RelZ", "RelX"]
    for col in numeric_cols:
        df_joined[col] = pd.to_numeric(df_joined[col], errors='coerce')

    # -------------------------------
    # 11. Calculate Runs Scored
    # -------------------------------
    df_joined['Runs Scored'] = np.maximum(
        0,
        np.maximum(
            df_joined['opponentCurrentRuns'].shift(-1) - df_joined['opponentCurrentRuns'],
            df_joined['currentRuns'].shift(-1) - df_joined['currentRuns'].fillna(0)
        )
    )

    # -------------------------------
    # 12. Convert gameDate to datetime
    # -------------------------------
    df_joined['gameDate'] = pd.to_datetime(df_joined['gameDate'])

    # -------------------------------
    # 13. Clean pitchResult, abbreviate
    # -------------------------------
    df_joined['clean_pitchResult'] = df_joined['pitchResult'].str.split(' on a').str[0].str.strip()
    event_abbreviations = {
        "Single": "1B",
        "Foul": "Foul",
        "Hit By Pitch": "HBP",
        "Strike Swinging": "SS",
        "Strikeout (Swinging)": "K",   # forward K for swinging
        "Strikeout (Looking)": "ꓘ",   # backward K for looking
        "Ball": "B",
        "Walk": "BB",
        "Home Run": "HR",
        "Double": "2B",
        "Fielder's Choice": "FC",
        "Triple": "3B",
        "Reached on Error": "ROE",
        "Sac Fly": "SF"
    }
    df_joined['clean_pitchResult'] = df_joined['clean_pitchResult'].map(event_abbreviations).fillna(df_joined['clean_pitchResult'])

    # -------------------------------
    # 14. Create Event_Desc
    # -------------------------------
    df_joined['Event_Desc'] = df_joined.apply(lambda row: (
        f"{row['balls']}-{row['strikes']} "
        f"{row['pitchTypeFull']}, "
        + (f"{int(row['Runs Scored'])} Run " if row['Runs Scored'] > 0 else '')
        + f"{row['clean_pitchResult']}, "
        + (
            "Bases Empty"
            if not (row['ManOn1st'] == 1 or row['ManOn2nd'] == 1 or row['ManOn3rd'] == 1)
            else "Runners on "
                 + " ".join(filter(None, [
                     "1st" if row['ManOn1st'] == 1 else '',
                     "2nd" if row['ManOn2nd'] == 1 else '',
                     "3rd" if row['ManOn3rd'] == 1 else ''
                 ]))
        )
        + f", {row['inn']} "
        f'{row["outs"]} Out'
    ), axis=1)

    # -------------------------------
    # 15. Mark Leadoff Batters
    # -------------------------------
    valid_leadoff_events = {"single", "double", "triple", "home_run", "walk", "hit_by_pitch"}

    # Step 1: Identify leadoff batters
    df_joined["inning_leadoff"] = df_joined.groupby(["gameDate", "inn"])["abNumInGame"].transform("min") == df_joined["abNumInGame"]

    # Step 2: Identify successful leadoff batters
    df_joined["inning_leadoff"] = df_joined["inning_leadoff"] & df_joined["event_category"].isin(valid_leadoff_events)

    # Step 3: If a leadoff batter succeeded in an inning, mark success for all rows in that inning
    df_joined["inning_leadoff_success"] = df_joined.groupby(["gameDate", "inn"])["inning_leadoff"].transform("max")

    # Convert True/False → 1/0
    df_joined["inning_leadoff"] = df_joined["inning_leadoff"].astype(int)
    df_joined["inning_leadoff_success"] = df_joined["inning_leadoff_success"].astype(int)

    df_joined['Taggedpitchtype'] = df_joined['pitchTypeFull']
    df_joined['Autopitchtype'] = df_joined['pitchTypeFull']

    # print("Feature engineering complete. Here's a preview:")
    # print(df_joined.head())
    # Ensure relevant columns are numeric
    numeric_columns = ["RelX", "HorzBrk", "IndVertBrk", "Vel"]
    for col in numeric_columns:
        if col in df_joined.columns:
            df_joined[col] = pd.to_numeric(df_joined[col], errors="coerce")

    # Drop rows with NaN in critical columns to avoid aggregation errors
    df_joined = df_joined.dropna(subset=numeric_columns)

    # # Determine pitcher handedness
    # df_hand = (
    #     df_joined.groupby("pitcherId", as_index=False)["RelX"].mean()
    #     .rename(columns={"RelX": "avg_RelX"})
    # )
    # df_hand["pitcher_hand"] = np.where(df_hand["avg_RelX"] > 0, "R", "L")

    # Merge handedness info back
    # df_joined = pd.merge(df_joined, df_hand[["pitcherId", "pitcher_hand"]], on="pitcherId", how="left")

    # Rename columns to standard references
    df_joined = df_joined.rename(columns={
        "Vel": "start_speed",
        "Spin": "spin_rate",
        "Extension": "extension",
        "RelZ": "z0",           # Release height
        "RelX": "x0",           # Release side
        "HorzBrk": "ax",        # Horizontal break
        "IndVertBrk": "az",     # Vertical break
        "PitchType": "pitch_type"
    })

    # Mirror for left-handed pitchers
    df_joined["ax"] = np.where(df_joined["pitcherHand"] == "L", -df_joined["ax"], df_joined["ax"])
    df_joined["x0"] = np.where(df_joined["pitcherHand"] == "L", -df_joined["x0"], df_joined["x0"])
    df_joined["is_fastball"] = df_joined["pitch_type"].isin(["FF", "FA", "SI"])
    # Most-used fastball logic
    fastball_types = ["FF", "SI", "FA"]
    df_joined["is_fastball"] = df_joined["pitch_type"].isin(["FF", "FA", "SI"])
    df_fb = df_joined[df_joined["pitch_type"].isin(fastball_types)].copy()

    # Group by (pitcherId, pitch_type), compute means & usage count
    df_agg = (
        df_fb.groupby(["pitcherId", "pitch_type"], as_index=False)
        .agg(
            avg_fastball_speed=("start_speed", "mean"),
            avg_fastball_az=("az", "mean"),
            avg_fastball_ax=("ax", "mean"),
            count=("start_speed", "count")
        )
    )

    # Sort by usage count, then avg_fastball_speed, descending
    df_agg = df_agg.sort_values(["count", "avg_fastball_speed"], ascending=[False, False])

    # Keep only the top row (most-used & fastest) per pitcherId
    df_agg = df_agg.drop_duplicates(subset=["pitcherId"], keep="first")

    # Merge back & compute diffs
    df_joined = pd.merge(
        df_joined,
        df_agg[["pitcherId", "avg_fastball_speed", "avg_fastball_az", "avg_fastball_ax"]],
        on="pitcherId",
        how="left"
    )

    df_joined["speed_diff"] = df_joined["start_speed"] - df_joined["avg_fastball_speed"]
    df_joined["az_diff"] = df_joined["az"] - df_joined["avg_fastball_az"]
    df_joined["ax_diff"] = df_joined["ax"] - df_joined["avg_fastball_ax"]
    df_joined["pitcher_hand"] = df_joined["pitcherHand"]

    return df_joined

#############################################
# 1) DEFINE HELPER FUNCTIONS
#############################################
def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """
    Feature engineering for a baseball dataset with columns:
      - relspeed      : pitch velocity
      - spinrate      : spin rate
      - extension     : release extension
      - relheight     : release height
      - relside       : release side (+ => typically R, - => L)
      - ax0           : horizontal pitch break
      - az0           : vertical pitch break
      - autopitchtype : pitch type (e.g., "Four-Seam", "Sinker", etc.)
      - pitcher       : pitcher identifier
      ... other columns as needed
    Steps:
      1) Determine pitcher handedness from average 'relside' (R if > 0, else L).
      2) Rename columns to standard references (start_speed, ax, az, etc.).
      3) Mirror horizontal release & break for left-handed pitchers.
      4) From fastball types ["Four-Seam","Sinker"], find the most-used fastball
         per (pitcher). If there's a tie, pick the one with the highest average speed.
      5) Merge those metrics back & compute diffs:
         - speed_diff = start_speed - avg_fastball_speed
         - az_diff    = az - avg_fastball_az
         - ax_diff    = ax - avg_fastball_ax
      6) Flip x0 sign (df["x0"] = df["x0"] * -1) at the end.
    """
    # 1) Keep only the columns we need
    needed_cols = [
        "pitcher",
        "relside",
        "relspeed",
        "spinrate",
        "extension",
        "relheight",
        "horzbreak",
        "inducedvertbreak",
        "autopitchtype",
        "pitchuid"
    ]
    df = df[needed_cols].copy()
    
    # 1) DETERMINE PITCHER HANDEDNESS
    df_hand = (
        df.groupby("pitcher", as_index=False)["relside"].mean()
          .rename(columns={"relside": "avg_side"})
    )
    df_hand["pitcher_hand"] = np.where(df_hand["avg_side"] > 0, "R", "L")

    # Merge handedness info back
    df = pd.merge(df, df_hand[["pitcher", "pitcher_hand"]], on="pitcher", how="left")

    # 2) RENAME COLUMNS
    df = df.rename(columns={
        "relspeed":      "start_speed",
        "spinrate":      "spin_rate",
        "extension":     "extension",
        "relheight":     "z0",
        "relside":       "x0",
        "horzbreak":           "ax",         
        "inducedvertbreak":           "az",         
        "autopitchtype": "pitch_type"
    })

    # 3) MIRROR FOR LEFT-HANDED PITCHERS
    df["ax"] = np.where(df["pitcher_hand"] == "L", -df["ax"], df["ax"])
    print(df["x0"].iloc[0])
    df["x0"] = np.where(df["pitcher_hand"] == "L", -df["x0"], df["x0"])

    # 4) MOST-USED FASTBALL LOGIC
    fastball_types = ["Four-Seam", "Sinker"]
    df_fb = df[df["pitch_type"].isin(fastball_types)].copy()

    df_agg = (
        df_fb.groupby(["pitcher", "pitch_type"], as_index=False)
             .agg(
                 avg_fastball_speed=("start_speed", "mean"),
                 avg_fastball_az=("az", "mean"),
                 avg_fastball_ax=("ax", "mean"),
                 count=("start_speed", "count")
             )
    )
    df_agg = df_agg.sort_values(["count", "avg_fastball_speed"], ascending=[False, False])
    df_agg = df_agg.drop_duplicates(subset=["pitcher"], keep="first")

    df = pd.merge(
        df,
        df_agg[["pitcher", "avg_fastball_speed", "avg_fastball_az", "avg_fastball_ax"]],
        on=["pitcher"],
        how="left"
    )

    df["speed_diff"] = df["start_speed"] - df["avg_fastball_speed"]
    df["az_diff"]    = df["az"] - df["avg_fastball_az"]
    df["ax_diff"]    = df["ax"] - df["avg_fastball_ax"]

    df["is_fastball"] = df["pitch_type"].isin(fastball_types)

    df["z0"] = df["z0"] * 12
    df["x0"] = df["x0"] * 12

    return df


def run_model_and_scale(df_for_model: pd.DataFrame) -> pd.DataFrame:
    """
    1) Load the trained model from disk.
    2) Predict using the engineered features.
    3) Add 'target' column.
    4) Apply z-score and tj_stuff_plus using pre-known baseline stats.
    Returns a new df with 'target', 'target_zscore', 'tj_stuff_plus'.
    """

    # -- LOAD MODEL
    # Get the directory of the currently running .py file
    # BASE_DIR = os.path.dirname(os.path.abspath(__file__))
    
    # Construct the path to your joblib file
    model_path = r"C:\Users\TrevorWhite\Downloads\NCAA_STUFF_PLUS_ALL.joblib"
    
    # Load the model
    model = joblib.load(model_path)

    # -- DEFINE FEATURES
    features = [
        "start_speed",
        "spin_rate",
        "extension",
        "az",
        "ax",
        "x0",
        "z0",
        "speed_diff",
        "az_diff",
        "ax_diff",
        "is_fastball"
    ]

    df_for_model[features] = df_for_model[features].apply(pd.to_numeric, errors='coerce')

    # -- MAKE PREDICTIONS
    predictions = model.predict(df_for_model[features])
    df_for_model["target"] = predictions

    # -- APPLY z-score & stuff-plus scaling
    target_mean_2023 = 0.011532333993710725
    target_std_2023  = 0.009399038486978739

    df_for_model["target_zscore"] = (
        (df_for_model["target"] - target_mean_2023) / target_std_2023
    )
    df_for_model["tj_stuff_plus"] = (
        100 - (df_for_model["target_zscore"] * 10)
    )

    whiff_model_path = r"C:\Users\TrevorWhite\Downloads\whiff_model_grouped_training.joblib"
    whiff_model = joblib.load(whiff_model_path)

    # --- PREDICT WHIFF (xWhiff) ---
    df_for_model["xWhiff"] = whiff_model.predict(df_for_model[features])

    return df_for_model


# Load the CSV file
file_path = "https://raw.githubusercontent.com/tdub29/streamlit-app-1/refs/heads/main/usd_baseball_TM_master_file.csv"
df = pd.read_csv(file_path)


df['Source'] = 'Preseason'

df.drop_duplicates(subset=['PitchUID'], inplace=True)

# Standardize column capitalization
df.columns = [col.strip().capitalize() for col in df.columns]

trufilepath = "https://raw.githubusercontent.com/tdub29/streamlit-app-1/refs/heads/main/USDPITCHINGYTD.csv"

Trumediadf = pd.read_csv(trufilepath)


Trumediadf['Source'] = 'Trumedia'




def process_relheight(df):
    # Define pitchers to exclude
    excluded_pitchers = ["Bunnell, Jack"]
    
    # Compute overall average Relheight per player
    player_overall_avg_relheight = (
        df.groupby('Pitcher')['Relheight']
        .mean()
        .reset_index()
        .rename(columns={'Relheight': 'player_overall_avg_relheight'})
    )
    
    # Filter out excluded pitchers
    df_non_excluded = df[~df['Pitcher'].isin(excluded_pitchers)].copy()
    
    # Compute average Relheight per player per date
    player_date_avg_relheight = (
        df_non_excluded.groupby(['Pitcher', 'Date'])['Relheight']
        .mean()
        .reset_index()
        .rename(columns={'Relheight': 'player_date_avg_relheight'})
    )
    
    # Merge overall and daily averages
    player_diff = pd.merge(
        player_date_avg_relheight, player_overall_avg_relheight, on='Pitcher', how='left'
    )
    
    # Compute the difference
    player_diff['diff_relheight'] = (
        player_diff['player_overall_avg_relheight'] - player_diff['player_date_avg_relheight']
    )
    
    # Compute average difference per date
    avg_diff_per_date = (
        player_diff.groupby('Date')['diff_relheight']
        .mean()
        .reset_index()
        .rename(columns={'diff_relheight': 'avg_diff_relheight'})
    )
    
    # Merge average difference back to main DataFrame
    df = pd.merge(df, avg_diff_per_date, on='Date', how='left')
    
    # Rename original 'Relheight' and create scaled 'Relheight'
    df.rename(columns={'Relheight': 'relheight_uncleaned'}, inplace=True)
    df['avg_diff_relheight'] = df['avg_diff_relheight'].fillna(0)
    df['Relheight'] = df['relheight_uncleaned'] + df['avg_diff_relheight']
    
    # Handle missing values
    df['Relheight'] = df['Relheight'].fillna(df['relheight_uncleaned'])
    
    return df

# Apply transformation only if Source is 'Trumedia'
if 'Source' in df.columns and (df['Source'] == 'Preseason').any():
    # Process Relheight
    df = process_relheight(df)
    
    # Create 'Swing' column based on Exitspeed and Pitchcall
    df['Swing'] = np.where(
        (df['Exitspeed'] > 0) | (df['Pitchcall'].str.contains('Swing|Foul', case=False, na=False)),
        'Swing',
        'Take'
    )

    # Create 'Contact' column based on Exitspeed
    df['Contact'] = np.where(df['Exitspeed'] > 0, 'Yes', 'No')

    # Create 'Whiff' column
    df['Whiff'] = np.where(
        (df['Swing'] == 'Swing') & (df['Contact'] == 'No'),
        1,  # Mark as 1 if both conditions are met
        0    # Otherwise, mark as 0
    ).astype(int)

    # Create 'Count' column combining Balls and Strikes
    df['Count'] = df['Balls'].astype(str) + '-' + df['Strikes'].astype(str)


df_for_model = df.copy()

# Rename columns to lowercase for the feature_engineering function
df_for_model.columns = [c.lower() for c in df_for_model.columns]

trudf_for_model = Trumediadf.copy()

trudf_for_model['Taggedpitchtype'] = trudf_for_model['pitchTypeFull']
trudf_for_model['Autopitchtype'] = trudf_for_model['pitchTypeFull']





In [ ]:
trudf_for_model.head(1)

In [ ]:
df.head(1)

In [ ]:

# Ensure the columns needed by feature_engineering exist
# (RelSpeed, RelHeight, RelSide, ax0, az0, AutoPitchType, Pitcher, SpinRate, Extension)
# If any are missing, you may need to handle that or rename them properly.

# 1) FEATURE ENGINEERING
df_for_model = feature_engineering(df_for_model)

# 2) RUN MODEL + SCALING
df_for_model = run_model_and_scale(df_for_model)

trudf_for_model = Trumedia_feature_engineering(trudf_for_model)

trudf_for_model.columns = [c.lower() for c in trudf_for_model.columns]

trudf_for_model = run_model_and_scale(trudf_for_model)




In [ ]:
if "pitchuid" in df_for_model.columns and "Pitchuid" in df.columns:
    # We select only the new columns from df_for_model we want to bring back
    merged_cols = ["pitchuid", "target", "target_zscore", "tj_stuff_plus", "xWhiff"]
    df = pd.merge(
        df, 
        df_for_model[merged_cols], 
        left_on="Pitchuid", right_on="pitchuid", 
        how="left"
    )
    # You might drop the duplicate "pitchuid" column from df
    df.drop(columns=["pitchuid"], inplace=True, errors="ignore")

In [ ]:
df.head(1)

In [ ]:
trudf_for_model.head(1)

In [ ]:
import pandas as pd

# 1) Load CSV file
trumedia_df = pd.read_csv(r"C:\Users\TrevorWhite\Downloads\trumediatotrackmannamejoin.csv")

# 2) Rename pitcherAbbrevName to pitcherabbrevname in 'trumedia_df'
trumedia_df.rename(columns={"pitcherAbbrevName": "pitcherabbrevname"}, inplace=True)

# 3) Drop 'Pitcher' from 'trudf_for_model' if it already exists
if "pitcher" in trudf_for_model.columns:
    trudf_for_model.drop(columns=["pitcher"], inplace=True)


trudf_for_model['Pitcherthrows'] = trudf_for_model['pitcher_hand'].apply(lambda x: 'Left' if x == 'L' else 'Right')

# 4) Merge 'trudf_for_model' with 'trumedia_df' to pull in the 'Pitcher' column
trudf_for_model = trudf_for_model.merge(
    trumedia_df[['pitcherabbrevname', 'Pitcher']], 
    on='pitcherabbrevname', 
    how='left'
)
trudf_for_model.columns = [col.strip().capitalize() for col in trudf_for_model.columns]
# 5) Rename columns in 'trudf_for_model'
trudf_for_model = trudf_for_model.rename(columns={
    "Start_speed": "Relspeed",
    "Spin_rate": "Spinrate",
    "extension": "Extension",
    "Z0": "Relheight",         # Release height
    "X0": "Relside",           # Release side
    "Ax": "Horzbreak",         # Horizontal break
    "Az": "Inducedvertbreak",  # Vertical break
    "pitchTypeFull": "Taggedpitchtype",
    "horzapprangle": "Horzapprangle",
    "vertapprangle": "Vertapprangle",
    "horzrelangle": "Horzrelangle",
    "vertrelangle": "Vertrelangle",
    "uniqPitchId": "pitchuid",
    "Pz": "Platelocheight",
    "Px": "Platelocside",
    "Batterhand": "Batterside",
    "Spindir": "Spinaxis",
    "balls": "Balls",
    "strikes": "Strikes",
    "Tj_stuff_plus": "tj_stuff_plus",
    "Launchang": "Angle",
    "Exitdir": "Direction",
    "Exitvelocity": "Exitspeed",
    "Xwhiff": "xWhiff"

})

# 6) If 'df' doesn't exist yet, define it as an empty DataFrame
if 'df' not in globals():
    df = pd.DataFrame()

# 7) Remove any duplicate columns in both DataFrames
df = df.loc[:, ~df.columns.duplicated()].copy()
trudf_for_model = trudf_for_model.loc[:, ~trudf_for_model.columns.duplicated()].copy()

# 8) Concatenate the two DataFrames
df = pd.concat([df, trudf_for_model], ignore_index=True, sort=False)

# 'df' now contains the appended data with the updated 'Pitcher' column


In [ ]:


# OPTIONAL: Merge the new columns (target, tj_stuff_plus) back into the original "df"
# so that you can reference them in your existing plots/tables if desired.
# We'll merge on a unique identifier you have (e.g., Pitchuid), if it exists in both.
# For demonstration, let's assume "pitchuid" (lowercase in df_for_model).

    
# Load arm angle CSV
armangle_path = "https://raw.githubusercontent.com/tdub29/streamlit-app-1/refs/heads/main/armangle_final_fall_usd.csv"
armangle_df = pd.read_csv(armangle_path)

# Merge arm angle data into df on 'Pitcher'
df = df.merge(armangle_df[['Pitcher', 'armangle_prediction']], on='Pitcher', how='left')

# df.dropna(subset=['Date'], inplace=True)
# df["datetime"] = pd.to_datetime(df["Date"], errors="coerce")
# df["Pitchno"] = pd.to_numeric(df["Pitchno"], errors="coerce")

# # 2) Add 12 hours (to shift from midnight to noon) 
# #    plus the minutes indicated by 'Time'
# df["datetime"] = (
#     df["datetime"]
#     + pd.to_timedelta(12, unit="h")         # shift to noon
#     + pd.to_timedelta(df["Pitchno"], unit="m")  # add the minutes from noon
# )


df['Pitchtype'] = df['Taggedpitchtype'].replace('Undefined', np.nan).fillna(df['Autopitchtype'])
df['Pitchtype'] = df['Pitchtype'].replace(['Four-Seam', 'FourSeamFastBall'], 'Fastball')
df['Pitchtype'] = df['Pitchtype'].replace(['ChangeUp'], 'Changeup')

# df.to_csv('streamlit_2024_fall_data.csv', index=False)


# Convert 'Tilt' column from HH:MM format to float (1:45 -> 1.75)
def convert_tilt_to_float(tilt_value):
    try:
        if isinstance(tilt_value, str) and ':' in tilt_value:
            hours, minutes = tilt_value.split(':')
            return float(hours) + float(minutes) / 60
        return np.nan
    except Exception as e:
        st.write(f"Error converting Tilt: {e}")
        return np.nan

# Apply Tilt conversion
df['Tilt_float'] = df['Tilt'].apply(convert_tilt_to_float)





# Identify in-zone pitches based on PlateLocSide and PlateLocHeight
df['Inzone'] = df.apply(
    lambda row: -0.83 <= row['Platelocside'] <= 0.83 and 1.5 <= row['Platelocheight'] <= 3.5, axis=1)

df['Comploc'] = df.apply(
    lambda row: -1.15 <= row['Platelocside'] <= 1.15 and 1.1 <= row['Platelocheight'] <= 3.9, axis=1)

# Define pitch categories based on initial pitch types
pitch_categories = {
    "Breaking Ball": ["Slider", "Curveball"],
    "Fastball": ["Fastball", "Four-Seam", "Sinker", "Cutter", "TwoSeamFastBall"],
    "Offspeed": ["ChangeUp", "Splitter"]
}

# Function to categorize pitch types into broader groups
def categorize_pitch_type(pitch_type):
    for category, pitches in pitch_categories.items():
        if pitch_type in pitches:
            return category
    return None

# Create a new column 'Pitchcategory' to categorize pitches
df['Pitchcategory'] = df['Pitchtype'].apply(categorize_pitch_type)

# Set up the color palette based on pitch type
pitch_types = df['Pitchtype'].unique()
palette = sns.color_palette('Set2', len(pitch_types))
color_map = {
        'Fastball': '#1f77b4',  # Blue
        'TwoSeamFastBall': '#1f77b4',  # Blue
        'Slider': '#ff7f0e',    # Orange
        'Curveball': '#2ca02c', # Green
        'ChangeUp': '#d62728',  # Red
        'Changeup': '#d62728',  # Red
        'Cutter': '#9467bd',    # Purple
        'Sinker': '#8c564b',    # Brown
        'Splitter': '#e377c2',  # Pink
        'Knuckleball': '#7f7f7f', # Gray
        'Other': '#7f7f7f' # Gray
    }

In [ ]:
df[df['Source'] != 'Preseason'].head()

In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon
import seaborn as sns
from datetime import datetime
import math
from matplotlib.patches import Arc
from matplotlib.patches import Ellipse
import joblib
import os
import subprocess
import sys
import lightgbm
import xgboost as xgb
import catboost
from matplotlib.colors import Normalize
# from pitcher_reports import generate_report



try:
    import sklearn
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn"])
    import sklearn

import sklearn

def Trumedia_feature_engineering(df):
    import pandas as pd
    import numpy as np

    # -------------------------------
    # 1. Define pitch mapping
    # -------------------------------
    pitch_mapping = {
        'FA': 'Fast',
        'CU': 'Break',
        'CH': 'Slow',
        'SL': 'Break',
        'SI': 'Fast',
        'FC': 'Fast',
        'UN': None,  # filter out
        'IN': None,  # filter out
        'FF': 'Fast',
        'FS': 'Slow',
        'KN': 'Slow'
    }

    # Filter out rows where pitchType maps to None
    df = df[df['pitchType'].isin([pt for pt, group in pitch_mapping.items() if group is not None])].copy()
    # Map pitch types to a new column 'pitchgroup'
    df['pitchgroup'] = df['pitchType'].map(pitch_mapping)

    df['Autopitchtype'] =  df['Taggedpitchtype']

    # -------------------------------
    # 2. Categorize the pitchResult
    # -------------------------------
    def categorize_event(event):
        """
        Categorize the pitchResult into a standardized event label.
        Bunt events are ignored (return None).
        """
        if not isinstance(event, str):
            return None
    
        event = event.lower()
        
        if "bunt" in event or "unknown" in event:
            return None
        elif "single" in event:
            return "single"
        elif "double play" in event:
            return "field_out"
        elif "double" in event:
            return "double"
        elif "triple" in event:
            return "triple"
        elif "home run" in event:
            return "home_run"
        elif "looking" in event:
            return "called_strike"
        elif "swinging" in event:
            return "swinging_strike"
        elif "hit by pitch" in event:
            return "hit_by_pitch"
        elif "walk" in event or "ball" in event:
            return "ball"
        elif "foul" in event:
            return "foul"
        elif ("line out" in event or "fly out" in event or "ground out" in event or
              "pop out" in event or "double play" in event or "reached on error" in event or
              "in play out" in event or "sac fly" in event or "fielder's choice" in event):
            return "field_out"
        elif "ball in the dirt" in event:
            return "ball"
        else:
            return "unknown"


    df['event_category'] = df['pitchResult'].apply(categorize_event)
    # Drop rows where event_category is None or unknown
    df = df[(df['event_category'].notna()) & (df['event_category'] != 'unknown')].copy()

    # -------------------------------
    # 3. Split 'count' into balls and strikes
    # -------------------------------
    df[['balls', 'strikes']] = df['count'].str.split('-', expand=True)
    df['balls'] = pd.to_numeric(df['balls'], errors='coerce')
    df['strikes'] = pd.to_numeric(df['strikes'], errors='coerce')

    # -------------------------------
    # 4. Merge with run_values
    # -------------------------------
    run_values = pd.read_csv("https://raw.githubusercontent.com/tdub29/streamlit-app-1/refs/heads/main/run_values.csv")
    run_values = run_values.rename(columns={"event": "event_category"})
    df_joined = pd.merge(
        df,
        run_values,
        on=["balls", "strikes", "event_category"],
        how="left"
    )



    # -------------------------------
    # 5. Create Binary 'win' Column
    # -------------------------------
    win_events = {'foul', 'called_strike', 'swinging_strike', 'field_out', 'strikeout'}
    df_joined['win'] = df_joined['event_category'].apply(lambda x: 1 if x in win_events else 0)

    # -------------------------------
    # 6. Adjust PX Orientation
    # -------------------------------
    df_joined['PX'] = df_joined['PX'] * -1

    # -------------------------------
    # 7. Create Binary Count Categories
    # -------------------------------
    df_joined['count_0_0'] = ((df_joined['balls'] == 0) & (df_joined['strikes'] == 0)).astype(int)
    df_joined['count_hitters'] = df_joined[['balls', 'strikes']].apply(
        lambda x: 1 if (x['balls'], x['strikes']) in [(1,0), (2,0), (3,0), (3,1)] else 0, axis=1
    )
    df_joined['count_pitchers'] = df_joined[['balls', 'strikes']].apply(
        lambda x: 1 if (x['balls'], x['strikes']) in [(0,2), (0,1), (1,2)] else 0, axis=1
    )
    df_joined['count_2k'] = ((df_joined['strikes'] == 2) & (df_joined['balls'] != 3)).astype(int)

    # -------------------------------
    # 8. Additional Binary Features
    # -------------------------------
    # 8.1 Strike
    strike_events = {"foul", "called_strike", "swinging_strike", "field_out", "strikeout",
                     "home_run", "triple", "double", "single"}
    df_joined['Strike'] = df_joined['event_category'].isin(strike_events)

    # # 8.2 Comploc
    # df_joined['Comploc'] = df_joined.apply(
    #     lambda row: -1.15 <= row['PX'] <= 1.15 and 1.1 <= row['PZ'] <= 3.9, axis=1
    # )

    # # 8.3 Inzone
    # df_joined['Inzone'] = df_joined.apply(
    #     lambda row: -0.83 <= row['PX'] <= 0.83 and 1.5 <= row['PZ'] <= 3.5, axis=1
    # )

    # 8.4 Swing
    swing_events = {"foul", "swinging_strike", "field_out", "home_run", "triple", "double", "single"}
    df_joined['Swing'] = df_joined['event_category'].isin(swing_events)

    # 8.5 Whiff
    df_joined['Whiff'] = df_joined['pitchResult'].str.lower().str.contains("swinging", na=False).astype(int)

    # -------------------------------
    # 9. Create delta_run_exp_squared
    # -------------------------------
    df_joined['delta_run_exp_squared'] = df_joined['delta_run_exp'].apply(lambda x:
        0.5 + (x - 0.5) * 0.5 if x > 0.5 else
        -0.5 + (x + 0.5) * 0.5 if x < -0.5 else
        0.2 + (x - 0.2) * 0.75 if x > 0.2 else
        -0.2 + (x + 0.2) * 0.75 if x < -0.2 else
        x * 2.5
    )

    # -------------------------------
    # 10. Convert Numeric Columns
    # -------------------------------
    numeric_cols = ["Vel", "delta_run_exp", "Extension", "HorzApprAngle", "VertApprAngle", 
                    "IndVertBrk", "HorzBrk", "RelZ", "RelX","HorzRelAngle", "VertRelAngle"]
    for col in numeric_cols:
        df_joined[col] = pd.to_numeric(df_joined[col], errors='coerce')

    # -------------------------------
    # 11. Calculate Runs Scored
    # -------------------------------
    df_joined['Runs Scored'] = np.maximum(
        0,
        np.maximum(
            df_joined['opponentCurrentRuns'].shift(-1) - df_joined['opponentCurrentRuns'],
            df_joined['currentRuns'].shift(-1) - df_joined['currentRuns'].fillna(0)
        )
    )

    # -------------------------------
    # 12. Convert gameDate to datetime
    # -------------------------------
    df_joined['gameDate'] = pd.to_datetime(df_joined['gameDate'], errors='coerce')


    # -------------------------------
    # 13. Clean pitchResult, abbreviate
    # -------------------------------
    df_joined['clean_pitchResult'] = df_joined['pitchResult'].str.split(' on a').str[0].str.strip()
    event_abbreviations = {
        "Single": "1B",
        "Foul": "Foul",
        "Hit By Pitch": "HBP",
        "Strike Swinging": "SS",
        "Strikeout (Swinging)": "K",   # forward K for swinging
        "Strikeout (Looking)": "ꓘ",   # backward K for looking
        "Ball": "B",
        "Walk": "BB",
        "Home Run": "HR",
        "Double": "2B",
        "Fielder's Choice": "FC",
        "Triple": "3B",
        "Reached on Error": "ROE",
        "Sac Fly": "SF"
    }
    df_joined['clean_pitchResult'] = df_joined['clean_pitchResult'].map(event_abbreviations).fillna(df_joined['clean_pitchResult'])

    # -------------------------------
    # 14. Create Event_Desc
    # -------------------------------
    df_joined['Event_Desc'] = df_joined.apply(lambda row: (
        f"{row['balls']}-{row['strikes']} "
        f"{row['pitchTypeFull']}, "
        + (f"{int(row['Runs Scored'])} Run " if row['Runs Scored'] > 0 else '')
        + f"{row['clean_pitchResult']}, "
        + (
            "Bases Empty"
            if not (row['ManOn1st'] == 1 or row['ManOn2nd'] == 1 or row['ManOn3rd'] == 1)
            else "Runners on "
                 + " ".join(filter(None, [
                     "1st" if row['ManOn1st'] == 1 else '',
                     "2nd" if row['ManOn2nd'] == 1 else '',
                     "3rd" if row['ManOn3rd'] == 1 else ''
                 ]))
        )
        + f", {row['inn']} "
        f'{row["outs"]} Out'
    ), axis=1)

    # -------------------------------
    # 15. Mark Leadoff Batters
    # -------------------------------
    valid_leadoff_events = {"single", "double", "triple", "home_run", "walk", "hit_by_pitch"}

    # Step 1: Identify leadoff batters
    df_joined["inning_leadoff"] = df_joined.groupby(["gameDate", "inn"])["abNumInGame"].transform("min") == df_joined["abNumInGame"]

    # Step 2: Identify successful leadoff batters
    df_joined["inning_leadoff"] = df_joined["inning_leadoff"] & df_joined["event_category"].isin(valid_leadoff_events)

    # Step 3: If a leadoff batter succeeded in an inning, mark success for all rows in that inning
    df_joined["inning_leadoff_success"] = df_joined.groupby(["gameDate", "inn"])["inning_leadoff"].transform("max")

    # Convert True/False → 1/0
    df_joined["inning_leadoff"] = df_joined["inning_leadoff"].astype(int)
    df_joined["inning_leadoff_success"] = df_joined["inning_leadoff_success"].fillna(0).astype(int)


    df_joined['Taggedpitchtype'] = df_joined['pitchTypeFull']
    df_joined['Autopitchtype'] = df_joined['pitchTypeFull']

    # print("Feature engineering complete. Here's a preview:")
    # print(df_joined.head())
    # Ensure relevant columns are numeric
    numeric_columns = ["RelX", "HorzBrk", "IndVertBrk", "Vel"]
    for col in numeric_columns:
        if col in df_joined.columns:
            df_joined[col] = pd.to_numeric(df_joined[col], errors="coerce")

    # Drop rows with NaN in critical columns to avoid aggregation errors
    df_joined = df_joined.dropna(subset=numeric_columns)

    # Determine pitcher handedness
    # df_hand = (
    #     df_joined.groupby("pitcherId", as_index=False)["RelX"].mean()
    #     .rename(columns={"RelX": "avg_RelX"})
    # )
    # df_hand["pitcher_hand"] = np.where(df_hand["avg_RelX"] > 0, "R", "L")

    # # Merge handedness info back
    # df_joined = pd.merge(df_joined, df_hand[["pitcherId", "pitcher_hand"]], on="pitcherId", how="left")

    # Rename columns to standard references
    df_joined = df_joined.rename(columns={
        "Vel": "start_speed",
        "Spin": "spin_rate",
        "Extension": "extension",
        "RelZ": "z0",           # Release height
        "RelX": "x0",           # Release side
        "HorzBrk": "ax",        # Horizontal break
        "IndVertBrk": "az",     # Vertical break
        "pitchType": "pitch_type"
    })

    # Mirror for left-handed pitchers
    df_joined["pitcher_hand"] = df_joined["pitcherHand"]
    df_joined["ax"] = np.where(df_joined["pitcher_hand"] == "L", -df_joined["ax"], df_joined["ax"])
    df_joined["x0"] = np.where(df_joined["pitcher_hand"] == "L", -df_joined["x0"], df_joined["x0"])
    df_joined["is_fastball"] = df_joined["pitch_type"].isin(["FF", "FA", "SI"])
    # Most-used fastball logic
    fastball_types = ["FF", "SI", "FA"]
    df_joined["is_fastball"] = df_joined["pitch_type"].isin(["FF", "FA", "SI"])
    df_fb = df_joined[df_joined["pitch_type"].isin(fastball_types)].copy()

    # Group by (pitcherId, pitch_type), compute means & usage count
    df_agg = (
        df_fb.groupby(["pitcherId", "pitch_type"], as_index=False)
        .agg(
            avg_fastball_speed=("start_speed", "mean"),
            avg_fastball_az=("az", "mean"),
            avg_fastball_ax=("ax", "mean"),
            count=("start_speed", "count")
        )
    )

    # Sort by usage count, then avg_fastball_speed, descending
    df_agg = df_agg.sort_values(["count", "avg_fastball_speed"], ascending=[False, False])

    # Keep only the top row (most-used & fastest) per pitcherId
    df_agg = df_agg.drop_duplicates(subset=["pitcherId"], keep="first")

    # Merge back & compute diffs
    df_joined = pd.merge(
        df_joined,
        df_agg[["pitcherId", "avg_fastball_speed", "avg_fastball_az", "avg_fastball_ax"]],
        on="pitcherId",
        how="left"
    )

    df_joined["speed_diff"] = df_joined["start_speed"] - df_joined["avg_fastball_speed"]
    df_joined["az_diff"] = df_joined["az"] - df_joined["avg_fastball_az"]
    df_joined["ax_diff"] = df_joined["ax"] - df_joined["avg_fastball_ax"]

    return df_joined

#############################################
# 1) DEFINE HELPER FUNCTIONS
#############################################
def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """
    Feature engineering for a baseball dataset with columns:
      - relspeed      : pitch velocity
      - spinrate      : spin rate
      - extension     : release extension
      - relheight     : release height
      - relside       : release side (+ => typically R, - => L)
      - ax0           : horizontal pitch break
      - az0           : vertical pitch break
      - autopitchtype : pitch type (e.g., "Four-Seam", "Sinker", etc.)
      - pitcher       : pitcher identifier
      ... other columns as needed
    Steps:
      1) Determine pitcher handedness from average 'relside' (R if > 0, else L).
      2) Rename columns to standard references (start_speed, ax, az, etc.).
      3) Mirror horizontal release & break for left-handed pitchers.
      4) From fastball types ["Four-Seam","Sinker"], find the most-used fastball
         per (pitcher). If there's a tie, pick the one with the highest average speed.
      5) Merge those metrics back & compute diffs:
         - speed_diff = start_speed - avg_fastball_speed
         - az_diff    = az - avg_fastball_az
         - ax_diff    = ax - avg_fastball_ax
      6) Flip x0 sign (df["x0"] = df["x0"] * -1) at the end.
    """
    # 1) Keep only the columns we need
    needed_cols = [
        "pitcher",
        "relside",
        "relspeed",
        "spinrate",
        "extension",
        "relheight",
        "horzbreak",
        "inducedvertbreak",
        "autopitchtype",
        "pitchuid"
    ]
    df = df[needed_cols].copy()
    
    # 1) DETERMINE PITCHER HANDEDNESS
    df_hand = (
        df.groupby("pitcher", as_index=False)["relside"].mean()
          .rename(columns={"relside": "avg_side"})
    )
    df_hand["pitcher_hand"] = np.where(df_hand["avg_side"] > 0, "R", "L")

    # Merge handedness info back
    df = pd.merge(df, df_hand[["pitcher", "pitcher_hand"]], on="pitcher", how="left")

    # 2) RENAME COLUMNS
    df = df.rename(columns={
        "relspeed":      "start_speed",
        "spinrate":      "spin_rate",
        "extension":     "extension",
        "relheight":     "z0",
        "relside":       "x0",
        "horzbreak":           "ax",         
        "inducedvertbreak":           "az",         
        "autopitchtype": "pitch_type"
    })

    # 3) MIRROR FOR LEFT-HANDED PITCHERS
    df["ax"] = np.where(df["pitcher_hand"] == "L", -df["ax"], df["ax"])
    # print(df["x0"].iloc[0])
    df["x0"] = np.where(df["pitcher_hand"] == "L", -df["x0"], df["x0"])

    # 4) MOST-USED FASTBALL LOGIC
    fastball_types = ["Four-Seam", "Sinker"]
    df_fb = df[df["pitch_type"].isin(fastball_types)].copy()

    df_agg = (
        df_fb.groupby(["pitcher", "pitch_type"], as_index=False)
             .agg(
                 avg_fastball_speed=("start_speed", "mean"),
                 avg_fastball_az=("az", "mean"),
                 avg_fastball_ax=("ax", "mean"),
                 count=("start_speed", "count")
             )
    )
    df_agg = df_agg.sort_values(["count", "avg_fastball_speed"], ascending=[False, False])
    df_agg = df_agg.drop_duplicates(subset=["pitcher"], keep="first")

    df = pd.merge(
        df,
        df_agg[["pitcher", "avg_fastball_speed", "avg_fastball_az", "avg_fastball_ax"]],
        on=["pitcher"],
        how="left"
    )

    df["speed_diff"] = df["start_speed"] - df["avg_fastball_speed"]
    df["az_diff"]    = df["az"] - df["avg_fastball_az"]
    df["ax_diff"]    = df["ax"] - df["avg_fastball_ax"]

    df["is_fastball"] = df["pitch_type"].isin(fastball_types)

    df["z0"] = df["z0"] * 12
    df["x0"] = df["x0"] * 12

    return df


def run_model_and_scale(df_for_model: pd.DataFrame) -> pd.DataFrame:
    """
    1) Load the trained model from disk.
    2) Predict using the engineered features.
    3) Add 'target' column.
    4) Apply z-score and tj_stuff_plus using pre-known baseline stats.
    Returns a new df with 'target', 'target_zscore', 'tj_stuff_plus'.
    """

    # -- LOAD MODEL
    # Get the directory of the currently running .py file
    # BASE_DIR = os.path.dirname(os.path.abspath(__file__))
    # Import joblib from the local repository
    import joblib  # Now import the local joblib
    
    # Construct the path to your joblib file
    model_path = r"C:\Users\TrevorWhite\Downloads\NCAA_STUFF_PLUS_ALL.joblib"
    
    # Load the model
    model = joblib.load(model_path)

    # -- DEFINE FEATURES
    features = [
        "start_speed",
        "spin_rate",
        "extension",
        "az",
        "ax",
        "x0",
        "z0",
        "speed_diff",
        "az_diff",
        "ax_diff",
        "is_fastball"
    ]

    df_for_model[features] = df_for_model[features].apply(pd.to_numeric, errors='coerce')

    # -- MAKE PREDICTIONS
    predictions = model.predict(df_for_model[features])
    df_for_model["target"] = predictions

    # -- APPLY z-score & stuff-plus scaling
    target_mean_2023 = 0.011532333993710725
    target_std_2023  = 0.009399038486978739

    df_for_model["target_zscore"] = (
        (df_for_model["target"] - target_mean_2023) / target_std_2023
    )
    df_for_model["tj_stuff_plus"] = (
        100 - (df_for_model["target_zscore"] * 10)
    )

    whiff_model_path = r"C:\Users\TrevorWhite\Downloads\whiff_model_grouped_training.joblib"
    whiff_model = joblib.load(whiff_model_path)

    # --- PREDICT WHIFF (xWhiff) ---
    df_for_model["xWhiff"] = whiff_model.predict(df_for_model[features])

    return df_for_model


# Load the CSV file
file_path = "https://raw.githubusercontent.com/tdub29/streamlit-app-1/refs/heads/main/usd_baseball_TM_master_file.csv"
df = pd.read_csv(file_path)


df['Source'] = 'Preseason'

df.drop_duplicates(subset=['PitchUID'], inplace=True)

# Standardize column capitalization
df.columns = [col.strip().capitalize() for col in df.columns]

# Load p_guys.csv
p_guys_df = pd.read_csv("https://raw.githubusercontent.com/tdub29/streamlit-app-1/refs/heads/main/p_guys.csv")
p_guys_df['gameDate'] = pd.to_datetime(p_guys_df['gameDate']).dt.strftime('%Y-%m-%d %H:%M:%S')
# Load USDPITCHINGYTD.csv
trufilepath = "https://raw.githubusercontent.com/tdub29/streamlit-app-1/refs/heads/main/USDPITCHINGYTD.csv"
usd_pitching_df = pd.read_csv(trufilepath)

# Combine them into Trumediadf
Trumediadf = pd.concat([p_guys_df, usd_pitching_df], ignore_index=True)


Trumediadf['Source'] = 'InSeason'

print("Check W_Par:", len(Trumediadf[Trumediadf['pitcherAbbrevName'] == 'W_Par']))
# Debug statements to inspect first dates from both dataframes
print("First date in p_guys_df:", p_guys_df['gameDate'].iloc[0])
print("First date in usd_pitching_df:", usd_pitching_df['gameDate'].iloc[0])



def process_relheight(df):
    # Define pitchers to exclude
    excluded_pitchers = ["Bunnell, Jack"]
    
    # Compute overall average Relheight per player
    player_overall_avg_relheight = (
        df.groupby('Pitcher')['Relheight']
        .mean()
        .reset_index()
        .rename(columns={'Relheight': 'player_overall_avg_relheight'})
    )
    
    # Filter out excluded pitchers
    df_non_excluded = df[~df['Pitcher'].isin(excluded_pitchers)].copy()
    
    # Compute average Relheight per player per date
    player_date_avg_relheight = (
        df_non_excluded.groupby(['Pitcher', 'Date'])['Relheight']
        .mean()
        .reset_index()
        .rename(columns={'Relheight': 'player_date_avg_relheight'})
    )
    
    # Merge overall and daily averages
    player_diff = pd.merge(
        player_date_avg_relheight, player_overall_avg_relheight, on='Pitcher', how='left'
    )
    
    # Compute the difference
    player_diff['diff_relheight'] = (
        player_diff['player_overall_avg_relheight'] - player_diff['player_date_avg_relheight']
    )
    
    # Compute average difference per date
    avg_diff_per_date = (
        player_diff.groupby('Date')['diff_relheight']
        .mean()
        .reset_index()
        .rename(columns={'diff_relheight': 'avg_diff_relheight'})
    )
    
    # Merge average difference back to main DataFrame
    df = pd.merge(df, avg_diff_per_date, on='Date', how='left')
    
    # Rename original 'Relheight' and create scaled 'Relheight'
    df.rename(columns={'Relheight': 'relheight_uncleaned'}, inplace=True)
    df['avg_diff_relheight'] = df['avg_diff_relheight'].fillna(0)
    df['Relheight'] = df['relheight_uncleaned'] + df['avg_diff_relheight']
    
    # Handle missing values
    df['Relheight'] = df['Relheight'].fillna(df['relheight_uncleaned'])
    
    return df

# Apply transformation only if Source is 'Trumedia'
if 'Source' in df.columns and (df['Source'] == 'Preseason').any():
    # Process Relheight
    df = process_relheight(df)
    
    # Create 'Swing' column based on Exitspeed and Pitchcall
    df['Swing'] = np.where(
        (df['Exitspeed'] > 0) | (df['Pitchcall'].str.contains('Swing|Foul', case=False, na=False)),
        'Swing',
        'Take'
    )

    # Create 'Contact' column based on Exitspeed
    df['Contact'] = np.where(df['Exitspeed'] > 0, 'Yes', 'No')

    # Create 'Whiff' column
    df['Whiff'] = np.where(
        (df['Swing'] == 'Swing') & (df['Contact'] == 'No'),
        1,  # Mark as 1 if both conditions are met
        0    # Otherwise, mark as 0
    ).astype(int)

    # Create 'Count' column combining Balls and Strikes
    df['Count'] = df['Balls'].astype(str) + '-' + df['Strikes'].astype(str)


df_for_model = df.copy()

# Rename columns to lowercase for the feature_engineering function
df_for_model.columns = [c.lower() for c in df_for_model.columns]

trudf_for_model = Trumediadf.copy()

trudf_for_model['Taggedpitchtype'] = trudf_for_model['pitchTypeFull']
trudf_for_model['Autopitchtype'] = trudf_for_model['pitchTypeFull']


# Ensure the columns needed by feature_engineering exist
# (RelSpeed, RelHeight, RelSide, ax0, az0, AutoPitchType, Pitcher, SpinRate, Extension)
# If any are missing, you may need to handle that or rename them properly.

# 1) FEATURE ENGINEERING
df_for_model = feature_engineering(df_for_model)

# 2) RUN MODEL + SCALING
df_for_model = run_model_and_scale(df_for_model)

trudf_for_model = Trumedia_feature_engineering(trudf_for_model)

trudf_for_model.columns = [c.lower() for c in trudf_for_model.columns]

trudf_for_model = run_model_and_scale(trudf_for_model)

print("Check W_Par post-engineering:", len(trudf_for_model[trudf_for_model['pitcherabbrevname'] == 'W_Par']))



if "pitchuid" in df_for_model.columns and "Pitchuid" in df.columns:
    # We select only the new columns from df_for_model we want to bring back
    merged_cols = ["pitchuid", "target", "target_zscore", "tj_stuff_plus", "xWhiff"]
    df = pd.merge(
        df, 
        df_for_model[merged_cols], 
        left_on="Pitchuid", right_on="pitchuid", 
        how="left"
    )
    # You might drop the duplicate "pitchuid" column from df
    df.drop(columns=["pitchuid"], inplace=True, errors="ignore")


import pandas as pd

# 1) Load the first CSV
trumedia_df = pd.read_csv("https://raw.githubusercontent.com/tdub29/streamlit-app-1/refs/heads/main/trumediatotrackmannamejoin.csv")

# 2) Rename 'pitcherAbbrevName' to 'pitcherabbrevname'
trumedia_df.rename(columns={"pitcherAbbrevName": "pitcherabbrevname"}, inplace=True)

# 3) Load the second CSV
# p_guys_df = pd.read_csv("https://raw.githubusercontent.com/tdub29/streamlit-app-1/refs/heads/main/p_guys.csv")

# p_guys_df.rename(columns={"pitcherAbbrevName": "pitcherabbrevname"}, inplace=True)

# # 4) Append second dataframe to the first
# trumedia_df = pd.concat([trumedia_df, p_guys_df], ignore_index=True)

# print("Check W_Par:", len(trumedia_df[trumedia_df['Pitcher'] == 'W_Par']))

# 3) Drop 'Pitcher' from 'trudf_for_model' if it already exists
if "pitcher" in trudf_for_model.columns:
    trudf_for_model.drop(columns=["pitcher"], inplace=True)


trudf_for_model['Pitcherthrows'] = trudf_for_model['pitcher_hand'].apply(lambda x: 'Left' if x == 'L' else 'Right')

# 4) Merge 'trudf_for_model' with 'trumedia_df' to pull in the 'Pitcher' column
trudf_for_model = trudf_for_model.merge(
    trumedia_df[['pitcherabbrevname', 'Pitcher']], 
    on='pitcherabbrevname', 
    how='left'
)
trudf_for_model.columns = [col.strip().capitalize() for col in trudf_for_model.columns]
# 5) Rename columns in 'trudf_for_model'
trudf_for_model = trudf_for_model.rename(columns={
    "Start_speed": "Relspeed",
    "Spin_rate": "Spinrate",
    "extension": "Extension",
    "Z0": "Relheight",         # Release height
    "X0": "Relside",           # Release side
    "Ax": "Horzbreak",         # Horizontal break
    "Az": "Inducedvertbreak",  # Vertical break
    "pitchTypeFull": "Taggedpitchtype",
    "horzapprangle": "Horzapprangle",
    "vertapprangle": "Vertapprangle",
    "horzrelangle": "Horzrelangle",
    "vertrelangle": "Vertrelangle",
    "uniqPitchId": "pitchuid",
    "Pz": "Platelocheight",
    "Px": "Platelocside",
    "Batterhand": "Batterside",
    "Spindir": "Spinaxis",
    "balls": "Balls",
    "strikes": "Strikes",
    "Tj_stuff_plus": "tj_stuff_plus",
    "Launchang": "Angle",
    "Exitdir": "Direction",
    "Exitvelocity": "Exitspeed",
    "Xwhiff": "xWhiff"

})
trudf_for_model['Date'] = pd.to_datetime(trudf_for_model['Date'], errors='coerce').dt.date

trudf_for_model[['Relheight', 'Relside']] /= 12
trudf_for_model['Batterside'] = trudf_for_model['Batterside'].map({'R': 'Right', 'L': 'Left'})


# print(trudf_for_model.head())

print("Check W_Par post-col renaming:", len(trudf_for_model[trudf_for_model['Pitcher'] == 'W_Par']))


# 6) If 'df' doesn't exist yet, define it as an empty DataFrame
if 'df' not in globals():
    df = pd.DataFrame()

# 7) Remove any duplicate columns in both DataFrames
df = df.loc[:, ~df.columns.duplicated()].copy()
trudf_for_model = trudf_for_model.loc[:, ~trudf_for_model.columns.duplicated()].copy()

# 8) Concatenate the two DataFrames
df = pd.concat([df, trudf_for_model], ignore_index=True, sort=False)

# 'df' now contains the appended data with the updated 'Pitcher' column




# OPTIONAL: Merge the new columns (target, tj_stuff_plus) back into the original "df"
# so that you can reference them in your existing plots/tables if desired.
# We'll merge on a unique identifier you have (e.g., Pitchuid), if it exists in both.
# For demonstration, let's assume "pitchuid" (lowercase in df_for_model).

    
# Load arm angle CSV
armangle_path = "https://raw.githubusercontent.com/tdub29/streamlit-app-1/refs/heads/main/armangle_final_fall_usd.csv"
armangle_df = pd.read_csv(armangle_path)

# Merge arm angle data into df on 'Pitcher'
df = df.merge(armangle_df[['Pitcher', 'armangle_prediction']], on='Pitcher', how='left')

# df.dropna(subset=['Date'], inplace=True)
# df["datetime"] = pd.to_datetime(df["Date"], errors="coerce")
# df["Pitchno"] = pd.to_numeric(df["Pitchno"], errors="coerce")

# # 2) Add 12 hours (to shift from midnight to noon) 
# #    plus the minutes indicated by 'Time'
# df["datetime"] = (
#     df["datetime"]
#     + pd.to_timedelta(12, unit="h")         # shift to noon
#     + pd.to_timedelta(df["Pitchno"], unit="m")  # add the minutes from noon
# )


df['Pitchtype'] = df['Taggedpitchtype'].replace('Undefined', np.nan).fillna(df['Autopitchtype'])
df['Pitchtype'] = df['Pitchtype'].replace(['Four-Seam', 'FourSeamFastBall'], 'Fastball')
df['Pitchtype'] = df['Pitchtype'].replace(['ChangeUp'], 'Changeup')

# df.to_csv('streamlit_2024_fall_data.csv', index=False)


# Convert 'Tilt' column from HH:MM format to float (1:45 -> 1.75)
def convert_tilt_to_float(tilt_value):
    try:
        if isinstance(tilt_value, str) and ':' in tilt_value:
            hours, minutes = tilt_value.split(':')
            return float(hours) + float(minutes) / 60
        return np.nan
    except Exception as e:
        st.write(f"Error converting Tilt: {e}")
        return np.nan

# Apply Tilt conversion
df['Tilt_float'] = df['Tilt'].apply(convert_tilt_to_float)





# Identify in-zone pitches based on PlateLocSide and PlateLocHeight
df['Inzone'] = df.apply(
    lambda row: -0.83 <= row['Platelocside'] <= 0.83 and 1.5 <= row['Platelocheight'] <= 3.5, axis=1)

df['Comploc'] = df.apply(
    lambda row: -1.15 <= row['Platelocside'] <= 1.15 and 1.1 <= row['Platelocheight'] <= 3.9, axis=1)

# Define pitch categories based on initial pitch types
pitch_categories = {
    "Breaking Ball": ["Slider", "Curveball"],
    "Fastball": ["Fastball", "Four-Seam", "Sinker", "Cutter", "TwoSeamFastBall"],
    "Offspeed": ["ChangeUp", "Splitter"]
}

# Function to categorize pitch types into broader groups
def categorize_pitch_type(pitch_type):
    for category, pitches in pitch_categories.items():
        if pitch_type in pitches:
            return category
    return None

# Create a new column 'Pitchcategory' to categorize pitches
df['Pitchcategory'] = df['Pitchtype'].apply(categorize_pitch_type)

df = df.copy()
df['PitcherPitchNo'] = df.groupby(['Pitcher', 'Date']).cumcount() + 1

# Set up the color palette based on pitch type
pitch_types = df['Pitchtype'].unique()
palette = sns.color_palette('Set2', len(pitch_types))
color_map = {
        'Fastball': '#1f77b4',  # Blue
        'TwoSeamFastBall': '#1f77b4',  # Blue
        'Slider': '#ff7f0e',    # Orange
        'Curveball': '#2ca02c', # Green
        'ChangeUp': '#d62728',  # Red
        'Changeup': '#d62728',  # Red
        'Cutter': '#9467bd',    # Purple
        'Sinker': '#8c564b',    # Brown
        'Splitter': '#e377c2',  # Pink
        'Knuckleball': '#7f7f7f', # Gray
        'Other': '#7f7f7f', # Gray
        'Undefined': '#7f7f7f',
        'FourSeamFastBall': '#1f77b4'
    }

# # ------------------------------------
# #  STREAMLIT SIDEBAR FILTERS (REPLACE)
# # ------------------------------------
# st.sidebar.header("Filter Options")

# # 1) Get unique pitchers and exclude "Bunnell, Jack"
# filtered_pitchers = df['Pitcher'].unique()
# filtered_pitchers = [pitcher for pitcher in filtered_pitchers if pitcher != "Bunnell, Jack"]
# # 2) Insert "All Pitchers" at the top (not as default selection)
# filtered_pitchers.insert(0, "All Pitchers")

# # 3) Selectbox for pitcher
# selected_pitcher = st.sidebar.selectbox("Select Pitcher", filtered_pitchers)

# # 6) Get unique sources and add filter
# sources_available = df['Source'].unique()
# selected_sources = st.sidebar.multiselect("Select Sources", sources_available, default=sources_available)

# # 4) Determine which dates to show:
# if selected_pitcher == "All Pitchers":
#     # If "All Pitchers" is selected, filter dates based on selected sources
#     dates_available = df[df['Source'].isin(selected_sources)]['Date'].unique()
# else:
#     # Otherwise, filter dates for the selected pitcher and selected sources
#     dates_available = df[(df['Pitcher'] == selected_pitcher) & (df['Source'].isin(selected_sources))]['Date'].unique()

# # 5) Multiselect for dates
# selected_dates = st.sidebar.multiselect("Select Dates", dates_available, default=dates_available)

# # 7) Filter the main DataFrame accordingly
# filtered_data = df[df['Date'].isin(selected_dates) & df['Source'].isin(selected_sources)]
# if selected_pitcher != "All Pitchers":
#     filtered_data = filtered_data[filtered_data['Pitcher'] == selected_pitcher]

In [ ]:
# ... existing code ...

# Add these pandas display options at the beginning of your notebook, after the imports
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', None)  # Width of the display in characters
pd.set_option('display.max_colwidth', None)  # Show full content of each column

# ... existing code ...
# df_transformed.head(250)

In [ ]:
print(df[df['Pitcher'] == 'Smith, Austin']['Date'].unique())

In [ ]:
# Load p_guys.csv
p_guys_df = pd.read_csv("https://raw.githubusercontent.com/tdub29/streamlit-app-1/refs/heads/main/p_guys.csv")

# Adjust the date format in p_guys_df
p_guys_df['gameDate'] = pd.to_datetime(p_guys_df['gameDate']).dt.strftime('%Y-%m-%d %H:%M:%S')

# Load USDPITCHINGYTD.csv
trufilepath = "https://raw.githubusercontent.com/tdub29/streamlit-app-1/refs/heads/main/USDPITCHINGYTD.csv"
Trumediadf = pd.read_csv(trufilepath)

# Combine them into Trumediadf
Trumediadf = pd.concat([Trumediadf, p_guys_df], ignore_index=True)


In [ ]:
Trumediadf['gameDate'].unique()

In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon
import seaborn as sns
from datetime import datetime
import math
from matplotlib.patches import Arc
from matplotlib.patches import Ellipse
import joblib
import os
import subprocess
import sys
import lightgbm
import xgboost as xgb
import catboost
from matplotlib.colors import Normalize
# from pitcher_reports import generate_report



try:
    import sklearn
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn"])
    import sklearn

import sklearn

def Trumedia_feature_engineering(df):
    import pandas as pd
    import numpy as np

    # -------------------------------
    # 1. Define pitch mapping
    # -------------------------------
    pitch_mapping = {
        'FA': 'Fast',
        'CU': 'Break',
        'CH': 'Slow',
        'SL': 'Break',
        'SI': 'Fast',
        'FC': 'Fast',
        'UN': None,  # filter out
        'IN': None,  # filter out
        'FF': 'Fast',
        'FS': 'Slow',
        'KN': 'Slow'
    }

    # Filter out rows where pitchType maps to None
    df = df[df['pitchType'].isin([pt for pt, group in pitch_mapping.items() if group is not None])].copy()
    # Map pitch types to a new column 'pitchgroup'
    df['pitchgroup'] = df['pitchType'].map(pitch_mapping)

    df['Autopitchtype'] =  df['Taggedpitchtype']

    # -------------------------------
    # 2. Categorize the pitchResult
    # -------------------------------
    def categorize_event(event):
        """
        Categorize the pitchResult into a standardized event label.
        Bunt events are ignored (return None).
        """
        if not isinstance(event, str):
            return None
    
        event = event.lower()
        
        if "bunt" in event or "unknown" in event:
            return None
        elif "single" in event:
            return "single"
        elif "double play" in event:
            return "field_out"
        elif "double" in event:
            return "double"
        elif "triple" in event:
            return "triple"
        elif "home run" in event:
            return "home_run"
        elif "looking" in event:
            return "called_strike"
        elif "swinging" in event:
            return "swinging_strike"
        elif "hit by pitch" in event:
            return "hit_by_pitch"
        elif "walk" in event or "ball" in event:
            return "ball"
        elif "foul" in event:
            return "foul"
        elif ("line out" in event or "fly out" in event or "ground out" in event or
              "pop out" in event or "double play" in event or "reached on error" in event or
              "in play out" in event or "sac fly" in event or "fielder's choice" in event):
            return "field_out"
        elif "ball in the dirt" in event:
            return "ball"
        else:
            return "unknown"


    df['event_category'] = df['pitchResult'].apply(categorize_event)
    # Drop rows where event_category is None or unknown
    df = df[(df['event_category'].notna()) & (df['event_category'] != 'unknown')].copy()

    # -------------------------------
    # 3. Split 'count' into balls and strikes
    # -------------------------------
    df[['balls', 'strikes']] = df['count'].str.split('-', expand=True)
    df['balls'] = pd.to_numeric(df['balls'], errors='coerce')
    df['strikes'] = pd.to_numeric(df['strikes'], errors='coerce')

    # -------------------------------
    # 4. Merge with run_values
    # -------------------------------
    run_values = pd.read_csv("https://raw.githubusercontent.com/tdub29/streamlit-app-1/refs/heads/main/run_values.csv")
    run_values = run_values.rename(columns={"event": "event_category"})
    df_joined = pd.merge(
        df,
        run_values,
        on=["balls", "strikes", "event_category"],
        how="left"
    )



    # -------------------------------
    # 5. Create Binary 'win' Column
    # -------------------------------
    win_events = {'foul', 'called_strike', 'swinging_strike', 'field_out', 'strikeout'}
    df_joined['win'] = df_joined['event_category'].apply(lambda x: 1 if x in win_events else 0)

    # -------------------------------
    # 6. Adjust PX Orientation
    # -------------------------------
    df_joined['PX'] = df_joined['PX'] * -1

    # -------------------------------
    # 7. Create Binary Count Categories
    # -------------------------------
    df_joined['count_0_0'] = ((df_joined['balls'] == 0) & (df_joined['strikes'] == 0)).astype(int)
    df_joined['count_hitters'] = df_joined[['balls', 'strikes']].apply(
        lambda x: 1 if (x['balls'], x['strikes']) in [(1,0), (2,0), (3,0), (3,1)] else 0, axis=1
    )
    df_joined['count_pitchers'] = df_joined[['balls', 'strikes']].apply(
        lambda x: 1 if (x['balls'], x['strikes']) in [(0,2), (0,1), (1,2)] else 0, axis=1
    )
    df_joined['count_2k'] = ((df_joined['strikes'] == 2) & (df_joined['balls'] != 3)).astype(int)

    # -------------------------------
    # 8. Additional Binary Features
    # -------------------------------
    # 8.1 Strike
    strike_events = {"foul", "called_strike", "swinging_strike", "field_out", "strikeout",
                     "home_run", "triple", "double", "single"}
    df_joined['Strike'] = df_joined['event_category'].isin(strike_events)

    # # 8.2 Comploc
    # df_joined['Comploc'] = df_joined.apply(
    #     lambda row: -1.15 <= row['PX'] <= 1.15 and 1.1 <= row['PZ'] <= 3.9, axis=1
    # )

    # # 8.3 Inzone
    # df_joined['Inzone'] = df_joined.apply(
    #     lambda row: -0.83 <= row['PX'] <= 0.83 and 1.5 <= row['PZ'] <= 3.5, axis=1
    # )

    # 8.4 Swing
    swing_events = {"foul", "swinging_strike", "field_out", "home_run", "triple", "double", "single"}
    df_joined['Swing'] = df_joined['event_category'].isin(swing_events)

    # 8.5 Whiff
    df_joined['Whiff'] = df_joined['pitchResult'].str.lower().str.contains("swinging", na=False).astype(int)

    # -------------------------------
    # 9. Create delta_run_exp_squared
    # -------------------------------
    df_joined['delta_run_exp_squared'] = df_joined['delta_run_exp'].apply(lambda x:
        0.5 + (x - 0.5) * 0.5 if x > 0.5 else
        -0.5 + (x + 0.5) * 0.5 if x < -0.5 else
        0.2 + (x - 0.2) * 0.75 if x > 0.2 else
        -0.2 + (x + 0.2) * 0.75 if x < -0.2 else
        x * 2.5
    )

    # -------------------------------
    # 10. Convert Numeric Columns
    # -------------------------------
    numeric_cols = ["Vel", "delta_run_exp", "Extension", "HorzApprAngle", "VertApprAngle", 
                    "IndVertBrk", "HorzBrk", "RelZ", "RelX","HorzRelAngle", "VertRelAngle"]
    for col in numeric_cols:
        df_joined[col] = pd.to_numeric(df_joined[col], errors='coerce')

    # -------------------------------
    # 11. Calculate Runs Scored
    # -------------------------------
    df_joined['Runs Scored'] = np.maximum(
        0,
        np.maximum(
            df_joined['opponentCurrentRuns'].shift(-1) - df_joined['opponentCurrentRuns'],
            df_joined['currentRuns'].shift(-1) - df_joined['currentRuns'].fillna(0)
        )
    )

    # -------------------------------
    # 12. Convert gameDate to datetime
    # -------------------------------
    df_joined['gameDate'] = pd.to_datetime(df_joined['gameDate'], errors='coerce')


    # -------------------------------
    # 13. Clean pitchResult, abbreviate
    # -------------------------------
    df_joined['clean_pitchResult'] = df_joined['pitchResult'].str.split(' on a').str[0].str.strip()
    event_abbreviations = {
        "Single": "1B",
        "Foul": "Foul",
        "Hit By Pitch": "HBP",
        "Strike Swinging": "SS",
        "Strikeout (Swinging)": "K",   # forward K for swinging
        "Strikeout (Looking)": "ꓘ",   # backward K for looking
        "Ball": "B",
        "Walk": "BB",
        "Home Run": "HR",
        "Double": "2B",
        "Fielder's Choice": "FC",
        "Triple": "3B",
        "Reached on Error": "ROE",
        "Sac Fly": "SF"
    }
    df_joined['clean_pitchResult'] = df_joined['clean_pitchResult'].map(event_abbreviations).fillna(df_joined['clean_pitchResult'])

    # -------------------------------
    # 14. Create Event_Desc
    # -------------------------------
    df_joined['Event_Desc'] = df_joined.apply(lambda row: (
        f"{row['balls']}-{row['strikes']} "
        f"{row['pitchTypeFull']}, "
        + (f"{int(row['Runs Scored'])} Run " if row['Runs Scored'] > 0 else '')
        + f"{row['clean_pitchResult']}, "
        + (
            "Bases Empty"
            if not (row['ManOn1st'] == 1 or row['ManOn2nd'] == 1 or row['ManOn3rd'] == 1)
            else "Runners on "
                 + " ".join(filter(None, [
                     "1st" if row['ManOn1st'] == 1 else '',
                     "2nd" if row['ManOn2nd'] == 1 else '',
                     "3rd" if row['ManOn3rd'] == 1 else ''
                 ]))
        )
        + f", {row['inn']} "
        f'{row["outs"]} Out'
    ), axis=1)

    # -------------------------------
    # 15. Mark Leadoff Batters
    # -------------------------------
    valid_leadoff_events = {"single", "double", "triple", "home_run", "walk", "hit_by_pitch"}

    # Step 1: Identify leadoff batters
    df_joined["inning_leadoff"] = df_joined.groupby(["gameDate", "inn"])["abNumInGame"].transform("min") == df_joined["abNumInGame"]

    # Step 2: Identify successful leadoff batters
    df_joined["inning_leadoff"] = df_joined["inning_leadoff"] & df_joined["event_category"].isin(valid_leadoff_events)

    # Step 3: If a leadoff batter succeeded in an inning, mark success for all rows in that inning
    df_joined["inning_leadoff_success"] = df_joined.groupby(["gameDate", "inn"])["inning_leadoff"].transform("max")

    # Convert True/False → 1/0
    df_joined["inning_leadoff"] = df_joined["inning_leadoff"].astype(int)
    df_joined["inning_leadoff_success"] = df_joined["inning_leadoff_success"].fillna(0).astype(int)


    df_joined['Taggedpitchtype'] = df_joined['pitchTypeFull']
    df_joined['Autopitchtype'] = df_joined['pitchTypeFull']

    # print("Feature engineering complete. Here's a preview:")
    # print(df_joined.head())
    # Ensure relevant columns are numeric
    numeric_columns = ["RelX", "HorzBrk", "IndVertBrk", "Vel"]
    for col in numeric_columns:
        if col in df_joined.columns:
            df_joined[col] = pd.to_numeric(df_joined[col], errors="coerce")

    # Drop rows with NaN in critical columns to avoid aggregation errors
    df_joined = df_joined.dropna(subset=numeric_columns)

    # Determine pitcher handedness
    # df_hand = (
    #     df_joined.groupby("pitcherId", as_index=False)["RelX"].mean()
    #     .rename(columns={"RelX": "avg_RelX"})
    # )
    # df_hand["pitcher_hand"] = np.where(df_hand["avg_RelX"] > 0, "R", "L")

    # # Merge handedness info back
    # df_joined = pd.merge(df_joined, df_hand[["pitcherId", "pitcher_hand"]], on="pitcherId", how="left")

    # Rename columns to standard references
    df_joined = df_joined.rename(columns={
        "Vel": "start_speed",
        "Spin": "spin_rate",
        "Extension": "extension",
        "RelZ": "z0",           # Release height
        "RelX": "x0",           # Release side
        "HorzBrk": "ax",        # Horizontal break
        "IndVertBrk": "az",     # Vertical break
        "pitchType": "pitch_type"
    })

    # Mirror for left-handed pitchers
    df_joined["pitcher_hand"] = df_joined["pitcherHand"]
    df_joined["ax"] = np.where(df_joined["pitcher_hand"] == "L", -df_joined["ax"], df_joined["ax"])
    df_joined["x0"] = np.where(df_joined["pitcher_hand"] == "L", -df_joined["x0"], df_joined["x0"])
    df_joined["is_fastball"] = df_joined["pitch_type"].isin(["FF", "FA", "SI"])
    # Most-used fastball logic
    fastball_types = ["FF", "SI", "FA"]
    df_joined["is_fastball"] = df_joined["pitch_type"].isin(["FF", "FA", "SI"])
    df_fb = df_joined[df_joined["pitch_type"].isin(fastball_types)].copy()

    # Group by (pitcherId, pitch_type), compute means & usage count
    df_agg = (
        df_fb.groupby(["pitcherId", "pitch_type"], as_index=False)
        .agg(
            avg_fastball_speed=("start_speed", "mean"),
            avg_fastball_az=("az", "mean"),
            avg_fastball_ax=("ax", "mean"),
            count=("start_speed", "count")
        )
    )

    # Sort by usage count, then avg_fastball_speed, descending
    df_agg = df_agg.sort_values(["count", "avg_fastball_speed"], ascending=[False, False])

    # Keep only the top row (most-used & fastest) per pitcherId
    df_agg = df_agg.drop_duplicates(subset=["pitcherId"], keep="first")

    # Merge back & compute diffs
    df_joined = pd.merge(
        df_joined,
        df_agg[["pitcherId", "avg_fastball_speed", "avg_fastball_az", "avg_fastball_ax"]],
        on="pitcherId",
        how="left"
    )

    df_joined["speed_diff"] = df_joined["start_speed"] - df_joined["avg_fastball_speed"]
    df_joined["az_diff"] = df_joined["az"] - df_joined["avg_fastball_az"]
    df_joined["ax_diff"] = df_joined["ax"] - df_joined["avg_fastball_ax"]

    return df_joined

#############################################
# 1) DEFINE HELPER FUNCTIONS
#############################################
def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """
    Feature engineering for a baseball dataset with columns:
      - relspeed      : pitch velocity
      - spinrate      : spin rate
      - extension     : release extension
      - relheight     : release height
      - relside       : release side (+ => typically R, - => L)
      - ax0           : horizontal pitch break
      - az0           : vertical pitch break
      - autopitchtype : pitch type (e.g., "Four-Seam", "Sinker", etc.)
      - pitcher       : pitcher identifier
      ... other columns as needed
    Steps:
      1) Determine pitcher handedness from average 'relside' (R if > 0, else L).
      2) Rename columns to standard references (start_speed, ax, az, etc.).
      3) Mirror horizontal release & break for left-handed pitchers.
      4) From fastball types ["Four-Seam","Sinker"], find the most-used fastball
         per (pitcher). If there's a tie, pick the one with the highest average speed.
      5) Merge those metrics back & compute diffs:
         - speed_diff = start_speed - avg_fastball_speed
         - az_diff    = az - avg_fastball_az
         - ax_diff    = ax - avg_fastball_ax
      6) Flip x0 sign (df["x0"] = df["x0"] * -1) at the end.
    """
    # 1) Keep only the columns we need
    needed_cols = [
        "pitcher",
        "relside",
        "relspeed",
        "spinrate",
        "extension",
        "relheight",
        "horzbreak",
        "inducedvertbreak",
        "autopitchtype",
        "pitchuid"
    ]
    df = df[needed_cols].copy()
    
    # 1) DETERMINE PITCHER HANDEDNESS
    df_hand = (
        df.groupby("pitcher", as_index=False)["relside"].mean()
          .rename(columns={"relside": "avg_side"})
    )
    df_hand["pitcher_hand"] = np.where(df_hand["avg_side"] > 0, "R", "L")

    # Merge handedness info back
    df = pd.merge(df, df_hand[["pitcher", "pitcher_hand"]], on="pitcher", how="left")

    # 2) RENAME COLUMNS
    df = df.rename(columns={
        "relspeed":      "start_speed",
        "spinrate":      "spin_rate",
        "extension":     "extension",
        "relheight":     "z0",
        "relside":       "x0",
        "horzbreak":           "ax",         
        "inducedvertbreak":           "az",         
        "autopitchtype": "pitch_type"
    })

    # 3) MIRROR FOR LEFT-HANDED PITCHERS
    df["ax"] = np.where(df["pitcher_hand"] == "L", -df["ax"], df["ax"])
    # print(df["x0"].iloc[0])
    df["x0"] = np.where(df["pitcher_hand"] == "L", -df["x0"], df["x0"])

    # 4) MOST-USED FASTBALL LOGIC
    fastball_types = ["Four-Seam", "Sinker"]
    df_fb = df[df["pitch_type"].isin(fastball_types)].copy()

    df_agg = (
        df_fb.groupby(["pitcher", "pitch_type"], as_index=False)
             .agg(
                 avg_fastball_speed=("start_speed", "mean"),
                 avg_fastball_az=("az", "mean"),
                 avg_fastball_ax=("ax", "mean"),
                 count=("start_speed", "count")
             )
    )
    df_agg = df_agg.sort_values(["count", "avg_fastball_speed"], ascending=[False, False])
    df_agg = df_agg.drop_duplicates(subset=["pitcher"], keep="first")

    df = pd.merge(
        df,
        df_agg[["pitcher", "avg_fastball_speed", "avg_fastball_az", "avg_fastball_ax"]],
        on=["pitcher"],
        how="left"
    )

    df["speed_diff"] = df["start_speed"] - df["avg_fastball_speed"]
    df["az_diff"]    = df["az"] - df["avg_fastball_az"]
    df["ax_diff"]    = df["ax"] - df["avg_fastball_ax"]

    df["is_fastball"] = df["pitch_type"].isin(fastball_types)

    df["z0"] = df["z0"] * 12
    df["x0"] = df["x0"] * 12

    return df


def run_model_and_scale(df_for_model: pd.DataFrame) -> pd.DataFrame:
    """
    1) Load the trained model from disk.
    2) Predict using the engineered features.
    3) Add 'target' column.
    4) Apply z-score and tj_stuff_plus using pre-known baseline stats.
    Returns a new df with 'target', 'target_zscore', 'tj_stuff_plus'.
    """

    # -- LOAD MODEL
    # Get the directory of the currently running .py file
    # BASE_DIR = os.path.dirname(os.path.abspath(__file__))
    # Import joblib from the local repository
    # Construct the path to your joblib file
    model_path = r"C:\Users\TrevorWhite\Downloads\NCAA_STUFF_PLUS_ALL.joblib"
    
    # Load the model
    model = joblib.load(model_path)


    # -- DEFINE FEATURES
    features = [
        "start_speed",
        "spin_rate",
        "extension",
        "az",
        "ax",
        "x0",
        "z0",
        "speed_diff",
        "az_diff",
        "ax_diff",
        "is_fastball"
    ]

    df_for_model[features] = df_for_model[features].apply(pd.to_numeric, errors='coerce')

    # -- MAKE PREDICTIONS
    predictions = model.predict(df_for_model[features])
    df_for_model["target"] = predictions

    # -- APPLY z-score & stuff-plus scaling
    target_mean_2023 = 0.011532333993710725
    target_std_2023  = 0.009399038486978739

    df_for_model["target_zscore"] = (
        (df_for_model["target"] - target_mean_2023) / target_std_2023
    )
    df_for_model["tj_stuff_plus"] = (
        100 - (df_for_model["target_zscore"] * 10)
    )

    whiff_model_path = r"C:\Users\TrevorWhite\Downloads\whiff_model_grouped_training.joblib"
    whiff_model = joblib.load(whiff_model_path)

    # --- PREDICT WHIFF (xWhiff) ---
    df_for_model["xWhiff"] = whiff_model.predict(df_for_model[features])

    return df_for_model


# Load the CSV file
file_path = "https://raw.githubusercontent.com/tdub29/hitterapp/refs/heads/main/USDHITTINGYTD.csv"
df = pd.read_csv(file_path)


df['Source'] = 'Preseason'

df.drop_duplicates(subset=['PitchUID'], inplace=True)

# Standardize column capitalization
df.columns = [col.strip().capitalize() for col in df.columns]

# Load p_guys.csv
p_guys_df = pd.read_csv("https://raw.githubusercontent.com/tdub29/streamlit-app-1/refs/heads/main/p_guys.csv")



# Load USDPITCHINGYTD.csv
trufilepath = "https://raw.githubusercontent.com/tdub29/streamlit-app-1/refs/heads/main/USDPITCHINGYTD.csv"
Trumediadf = pd.read_csv(trufilepath)

# Ensure p_guys_df['gameDate'] is datetime
p_guys_df['gameDate'] = pd.to_datetime(p_guys_df['gameDate']).dt.strftime('%Y-%m-%d %H:%M:%S')

# Ensure Trumediadf['gameDate'] is datetime
Trumediadf['gameDate'] = pd.to_datetime(Trumediadf['gameDate']).dt.strftime('%Y-%m-%d %H:%M:%S')

# Add 'insznsource' column to identify the source of each row
p_guys_df['insznsource'] = 'p_guys'
Trumediadf['insznsource'] = 'trumedia'

# Debug statements to inspect first dates from both dataframes
print("First date in p_guys_df:", p_guys_df['gameDate'].iloc[0])
print("First date in Trumediadf:", Trumediadf['gameDate'].iloc[0])

# Combine them into Trumediadf
Trumediadf = pd.concat([p_guys_df, Trumediadf], ignore_index=True)

# Check first date for both insznsrc
print("First date in p_guys_df after concatenation:", Trumediadf[Trumediadf['insznsource'] == 'p_guys']['gameDate'].iloc[0])
print("First date in Trumediadf after concatenation:", Trumediadf[Trumediadf['insznsource'] == 'trumedia']['gameDate'].iloc[0])



Trumediadf['Source'] = 'InSeason'




def process_relheight(df):
    # Define pitchers to exclude
    excluded_pitchers = ["Bunnell, Jack"]
    
    # Compute overall average Relheight per player
    player_overall_avg_relheight = (
        df.groupby('Pitcher')['Relheight']
        .mean()
        .reset_index()
        .rename(columns={'Relheight': 'player_overall_avg_relheight'})
    )
    
    # Filter out excluded pitchers
    df_non_excluded = df[~df['Pitcher'].isin(excluded_pitchers)].copy()
    
    # Compute average Relheight per player per date
    player_date_avg_relheight = (
        df_non_excluded.groupby(['Pitcher', 'Date'])['Relheight']
        .mean()
        .reset_index()
        .rename(columns={'Relheight': 'player_date_avg_relheight'})
    )
    
    # Merge overall and daily averages
    player_diff = pd.merge(
        player_date_avg_relheight, player_overall_avg_relheight, on='Pitcher', how='left'
    )
    
    # Compute the difference
    player_diff['diff_relheight'] = (
        player_diff['player_overall_avg_relheight'] - player_diff['player_date_avg_relheight']
    )
    
    # Compute average difference per date
    avg_diff_per_date = (
        player_diff.groupby('Date')['diff_relheight']
        .mean()
        .reset_index()
        .rename(columns={'diff_relheight': 'avg_diff_relheight'})
    )
    
    # Merge average difference back to main DataFrame
    df = pd.merge(df, avg_diff_per_date, on='Date', how='left')
    
    # Rename original 'Relheight' and create scaled 'Relheight'
    df.rename(columns={'Relheight': 'relheight_uncleaned'}, inplace=True)
    df['avg_diff_relheight'] = df['avg_diff_relheight'].fillna(0)
    df['Relheight'] = df['relheight_uncleaned'] + df['avg_diff_relheight']
    
    # Handle missing values
    df['Relheight'] = df['Relheight'].fillna(df['relheight_uncleaned'])
    
    return df

# Apply transformation only if Source is 'Trumedia'
if 'Source' in df.columns and (df['Source'] == 'Preseason').any():
    # Process Relheight
    df = process_relheight(df)
    
    # Create 'Swing' column based on Exitspeed and Pitchcall
    df['Swing'] = np.where(
        (df['Exitspeed'] > 0) | (df['Pitchcall'].str.contains('Swing|Foul', case=False, na=False)),
        'Swing',
        'Take'
    )

    # Create 'Contact' column based on Exitspeed
    df['Contact'] = np.where(df['Exitspeed'] > 0, 'Yes', 'No')

    # Create 'Whiff' column
    df['Whiff'] = np.where(
        (df['Swing'] == 'Swing') & (df['Contact'] == 'No'),
        1,  # Mark as 1 if both conditions are met
        0    # Otherwise, mark as 0
    ).astype(int)

    # Create 'Count' column combining Balls and Strikes
    df['Count'] = df['Balls'].astype(str) + '-' + df['Strikes'].astype(str)


df_for_model = df.copy()

# Rename columns to lowercase for the feature_engineering function
df_for_model.columns = [c.lower() for c in df_for_model.columns]

trudf_for_model = Trumediadf.copy()

trudf_for_model['Taggedpitchtype'] = trudf_for_model['pitchTypeFull']
trudf_for_model['Autopitchtype'] = trudf_for_model['pitchTypeFull']


# Ensure the columns needed by feature_engineering exist
# (RelSpeed, RelHeight, RelSide, ax0, az0, AutoPitchType, Pitcher, SpinRate, Extension)
# If any are missing, you may need to handle that or rename them properly.

# 1) FEATURE ENGINEERING
df_for_model = feature_engineering(df_for_model)

# 2) RUN MODEL + SCALING
df_for_model = run_model_and_scale(df_for_model)

trudf_for_model = Trumedia_feature_engineering(trudf_for_model)

trudf_for_model.columns = [c.lower() for c in trudf_for_model.columns]

trudf_for_model = run_model_and_scale(trudf_for_model)


# Check first date for both insznsrc
print("First date in p_guys_df after concatenation:", trudf_for_model[trudf_for_model['insznsource'] == 'p_guys']['gamedate'].iloc[0])
print("First date in Trumediadf after concatenation:", trudf_for_model[trudf_for_model['insznsource'] == 'trumedia']['gamedate'].iloc[0])




if "pitchuid" in df_for_model.columns and "Pitchuid" in df.columns:
    # We select only the new columns from df_for_model we want to bring back
    merged_cols = ["pitchuid", "target", "target_zscore", "tj_stuff_plus", "xWhiff"]
    df = pd.merge(
        df, 
        df_for_model[merged_cols], 
        left_on="Pitchuid", right_on="pitchuid", 
        how="left"
    )
    # You might drop the duplicate "pitchuid" column from df
    df.drop(columns=["pitchuid"], inplace=True, errors="ignore")


import pandas as pd

# 1) Load the first CSV
trumedia_df = pd.read_csv("https://raw.githubusercontent.com/tdub29/streamlit-app-1/refs/heads/main/trumediatotrackmannamejoin.csv")

# 2) Rename 'pitcherAbbrevName' to 'pitcherabbrevname'
trumedia_df.rename(columns={"pitcherAbbrevName": "pitcherabbrevname"}, inplace=True)

# # 3) Load the second CSV
# p_guys_df = pd.read_csv("https://raw.githubusercontent.com/tdub29/streamlit-app-1/refs/heads/main/p_guys.csv")

# # 4) Append second dataframe to the first
# trumedia_df = pd.concat([trumedia_df, p_guys_df], ignore_index=True)

# 3) Drop 'Pitcher' from 'trudf_for_model' if it already exists
if "pitcher" in trudf_for_model.columns:
    trudf_for_model.drop(columns=["pitcher"], inplace=True)


trudf_for_model['Pitcherthrows'] = trudf_for_model['pitcher_hand'].apply(lambda x: 'Left' if x == 'L' else 'Right')

# 4) Merge 'trudf_for_model' with 'trumedia_df' to pull in the 'Pitcher' column
trudf_for_model = trudf_for_model.merge(
    trumedia_df[['pitcherabbrevname', 'Pitcher']], 
    on='pitcherabbrevname', 
    how='left'
)
trudf_for_model.columns = [col.strip().capitalize() for col in trudf_for_model.columns]
# 5) Rename columns in 'trudf_for_model'
trudf_for_model = trudf_for_model.rename(columns={
    "Start_speed": "Relspeed",
    "Spin_rate": "Spinrate",
    "extension": "Extension",
    "Z0": "Relheight",         # Release height
    "X0": "Relside",           # Release side
    "Ax": "Horzbreak",         # Horizontal break
    "Az": "Inducedvertbreak",  # Vertical break
    "pitchTypeFull": "Taggedpitchtype",
    "horzapprangle": "Horzapprangle",
    "vertapprangle": "Vertapprangle",
    "horzrelangle": "Horzrelangle",
    "vertrelangle": "Vertrelangle",
    "uniqPitchId": "pitchuid",
    "Pz": "Platelocheight",
    "Px": "Platelocside",
    "Batterhand": "Batterside",
    "Spindir": "Spinaxis",
    "balls": "Balls",
    "strikes": "Strikes",
    "Tj_stuff_plus": "tj_stuff_plus",
    "Launchang": "Angle",
    "Exitdir": "Direction",
    "Exitvelocity": "Exitspeed",
    "Xwhiff": "xWhiff"

})

print("First date in p_guys_df after last conv:", trudf_for_model[trudf_for_model['Insznsource'] == 'p_guys']['Date'].iloc[0],
      "| Type:", type(trudf_for_model[trudf_for_model['Insznsource'] == 'p_guys']['Date'].iloc[0]))

print("First date in Trumediadf after last conv:", trudf_for_model[trudf_for_model['Insznsource'] == 'trumedia']['Date'].iloc[0],
      "| Type:", type(trudf_for_model[trudf_for_model['Insznsource'] == 'trumedia']['Date'].iloc[0]))



# 1. Create a new column to store parsed datetime
trudf_for_model['parsed_datetime'] = pd.NaT

# 2. Mask each source
mask_pguys = trudf_for_model['Insznsource'] == 'p_guys'
mask_trumedia = trudf_for_model['Insznsource'] == 'trumedia'

# 3. Parse separately
trudf_for_model.loc[mask_pguys, 'parsed_datetime'] = pd.to_datetime(
    trudf_for_model.loc[mask_pguys, 'Date'],
    format='%m/%d/%Y %H:%M',
    errors='coerce'
)

trudf_for_model.loc[mask_trumedia, 'parsed_datetime'] = pd.to_datetime(
    trudf_for_model.loc[mask_trumedia, 'Date'],
    format='%Y-%m-%d %H:%M:%S',
    errors='coerce'
)

# 4. Then extract date (safe to use .dt now)
trudf_for_model['Date'] = trudf_for_model['parsed_datetime'].dt.date

# Optional: drop the temporary column
trudf_for_model.drop(columns=['parsed_datetime'], inplace=True)




trudf_for_model[['Relheight', 'Relside']] /= 12
trudf_for_model['Batterside'] = trudf_for_model['Batterside'].map({'R': 'Right', 'L': 'Left'})
print("First date in p_guys_df after last conv:", trudf_for_model[trudf_for_model['Insznsource'] == 'p_guys']['Date'].iloc[0])
print("First date in Trumediadf after last conv:", trudf_for_model[trudf_for_model['Insznsource'] == 'trumedia']['Date'].iloc[0])




# 6) If 'df' doesn't exist yet, define it as an empty DataFrame
if 'df' not in globals():
    df = pd.DataFrame()

# 7) Remove any duplicate columns in both DataFrames
df = df.loc[:, ~df.columns.duplicated()].copy()
trudf_for_model = trudf_for_model.loc[:, ~trudf_for_model.columns.duplicated()].copy()

# 8) Concatenate the two DataFrames
df = pd.concat([df, trudf_for_model], ignore_index=True, sort=False)

# 'df' now contains the appended data with the updated 'Pitcher' column




# OPTIONAL: Merge the new columns (target, tj_stuff_plus) back into the original "df"
# so that you can reference them in your existing plots/tables if desired.
# We'll merge on a unique identifier you have (e.g., Pitchuid), if it exists in both.
# For demonstration, let's assume "pitchuid" (lowercase in df_for_model).

    
# Load arm angle CSV
armangle_path = "https://raw.githubusercontent.com/tdub29/streamlit-app-1/refs/heads/main/armangle_final_fall_usd.csv"
armangle_df = pd.read_csv(armangle_path)

# Merge arm angle data into df on 'Pitcher'
df = df.merge(armangle_df[['Pitcher', 'armangle_prediction']], on='Pitcher', how='left')

# df.dropna(subset=['Date'], inplace=True)
# df["datetime"] = pd.to_datetime(df["Date"], errors="coerce")
# df["Pitchno"] = pd.to_numeric(df["Pitchno"], errors="coerce")

# # 2) Add 12 hours (to shift from midnight to noon) 
# #    plus the minutes indicated by 'Time'
# df["datetime"] = (
#     df["datetime"]
#     + pd.to_timedelta(12, unit="h")         # shift to noon
#     + pd.to_timedelta(df["Pitchno"], unit="m")  # add the minutes from noon
# )


df['Pitchtype'] = df['Taggedpitchtype'].replace('Undefined', np.nan).fillna(df['Autopitchtype'])
df['Pitchtype'] = df['Pitchtype'].replace(['Four-Seam', 'FourSeamFastBall'], 'Fastball')
df['Pitchtype'] = df['Pitchtype'].replace(['ChangeUp'], 'Changeup')

# df.to_csv('streamlit_2024_fall_data.csv', index=False)


# Convert 'Tilt' column from HH:MM format to float (1:45 -> 1.75)
def convert_tilt_to_float(tilt_value):
    try:
        if isinstance(tilt_value, str) and ':' in tilt_value:
            hours, minutes = tilt_value.split(':')
            return float(hours) + float(minutes) / 60
        return np.nan
    except Exception as e:
        # st.write(f"Error converting Tilt: {e}")
        return np.nan

# Apply Tilt conversion
df['Tilt_float'] = df['Tilt'].apply(convert_tilt_to_float)





# Identify in-zone pitches based on PlateLocSide and PlateLocHeight
df['Inzone'] = df.apply(
    lambda row: -0.83 <= row['Platelocside'] <= 0.83 and 1.5 <= row['Platelocheight'] <= 3.5, axis=1)

df['Comploc'] = df.apply(
    lambda row: -1.15 <= row['Platelocside'] <= 1.15 and 1.1 <= row['Platelocheight'] <= 3.9, axis=1)

# Define pitch categories based on initial pitch types
pitch_categories = {
    "Breaking Ball": ["Slider", "Curveball"],
    "Fastball": ["Fastball", "Four-Seam", "Sinker", "Cutter", "TwoSeamFastBall"],
    "Offspeed": ["ChangeUp", "Splitter"]
}

# Function to categorize pitch types into broader groups
def categorize_pitch_type(pitch_type):
    for category, pitches in pitch_categories.items():
        if pitch_type in pitches:
            return category
    return None

# Create a new column 'Pitchcategory' to categorize pitches
df['Pitchcategory'] = df['Pitchtype'].apply(categorize_pitch_type)

df = df.copy()
df['PitcherPitchNo'] = df.groupby(['Pitcher', 'Date']).cumcount() + 1

# Set up the color palette based on pitch type
pitch_types = df['Pitchtype'].unique()
palette = sns.color_palette('Set2', len(pitch_types))
color_map = {
        'Fastball': '#1f77b4',  # Blue
        'TwoSeamFastBall': '#1f77b4',  # Blue
        'Slider': '#ff7f0e',    # Orange
        'Curveball': '#2ca02c', # Green
        'ChangeUp': '#d62728',  # Red
        'Changeup': '#d62728',  # Red
        'Cutter': '#9467bd',    # Purple
        'Sinker': '#8c564b',    # Brown
        'Splitter': '#e377c2',  # Pink
        'Knuckleball': '#7f7f7f', # Gray
        'Other': '#7f7f7f', # Gray
        'Undefined': '#7f7f7f',
        'FourSeamFastBall': '#1f77b4'
    }

# ------------

In [ ]:
trudf_for_model.head()

In [ ]:
print(df[df['Pitcher'] == 'Smith, Austin']['Date'].unique())

##HITTER APP


In [ ]:
import sys
import subprocess
import streamlit as st
##v2

# Install scikit-learn if it's not already installed
try:
    import sklearn
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn"])
    import sklearn

import sklearn
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
import matplotlib.lines as mlines
import xgboost as xgb
from statsmodels.nonparametric.kernel_regression import KernelReg
try:
    from scipy.stats import gaussian_kde
except ImportError as e:
    print(f"Error importing gaussian_kde: {e}")
import matplotlib as mpl

############################################
# Load Machine Learning Models
############################################
booster = xgb.Booster()
booster.load_model("C:/Users/TrevorWhite/OneDrive - Good360/Documents/Python Scripts/xSLG_model.json")

best_model = xgb.XGBRegressor()
best_model._Booster = booster  # Assign the booster to the regressor

# Load the No-Swing model (JSON)
no_swing_booster = xgb.Booster()
no_swing_booster.load_model("C:/Users/TrevorWhite/Downloads/model_no_swing.json")
model_no_swing = xgb.XGBRegressor()
model_no_swing._Booster = no_swing_booster

# Load the Swing model (JSON)
swing_booster = xgb.Booster()
swing_booster.load_model("C:/Users/TrevorWhite/Downloads/model_swing.json")
model_swing = xgb.XGBRegressor()
model_swing._Booster = swing_booster

############################################
# Custom Color Palette and Plot Styling
############################################
kde_min = '#236abe'
kde_mid = '#fefefe'
kde_max = '#a9373b'

kde_palette = (sns.color_palette(f'blend:{kde_min},{kde_mid}', n_colors=1001)[:-1] +
               sns.color_palette(f'blend:{kde_mid},{kde_max}', n_colors=1001)[:-1])

pl_white = '#FEFEFE'
pl_background = '#162B50'
pl_text = '#72a3f7'
pl_line_color = '#293a6b'

sns.set_theme(
    style={
        'axes.edgecolor': pl_white,
        'axes.facecolor': pl_white,
        'axes.labelcolor': pl_white,
        'xtick.color': pl_white,
        'ytick.color': pl_white,
        'figure.facecolor': pl_background,
        'grid.color': pl_background,
        'grid.linestyle': '-',
        'legend.facecolor': pl_background,
        'text.color': pl_white
    }
)

############################################
# Load CSV With New Column Names and Rename Accordingly
############################################
# (Update file_path to your new CSV source if needed)
file_path = "https://raw.githubusercontent.com/tdub29/hitterapp/refs/heads/main/USDHITTINGYTD.csv"
df = pd.read_csv(file_path)

# Use a mapping dictionary to rename new columns to the ones the code expects:
# (All keys are assumed case-insensitive; adjust if needed.)
rename_mapping = {
    "gamedate": "Date",
    "batter": "Batter",
    "pitcher": "Pitcher",
    "exitvelocity": "Exitspeed",   # use the provided exit velocity
    "launchang": "Angle",
    "x": "Platelocside",
    "y": "Platelocheight",
    "pitchoutcome": "Pitchcall",
    "pitchresult": "Playresult",
    "batterhand": "Batterside",
    "pitcherhand": "Pitcherhand",
    "uniqpitchid": "Pitchuid",
    "pitchtypefull": "Autopitchtype",
    "count": "Count",
    "abnumingame": "Paofinning",
    "inn": "Inning",
    "exitdir": "Direction",
    "dist": "Distance"
}

# Convert column names to lower-case (strip spaces) for matching keys
df.columns = [col.strip() for col in df.columns]
# Prepare a dictionary using lower() for matching
current_cols = {col.lower(): col for col in df.columns}
final_mapping = {}
for key, new_name in rename_mapping.items():
    if key in current_cols:
        final_mapping[current_cols[key]] = new_name

df.rename(columns=final_mapping, inplace=True)

############################################
# Create Derived Columns
############################################
# Split "Count" (e.g., "0-0") into numeric Balls and Strikes.
# (Assumes count is in the format "balls-strikes".)
df[['Balls', 'Strikes']] = df['Count'].str.split('-', expand=True).astype(int)
# Now rename these to lower-case as used later:
df.rename(columns={"Balls": "balls", "Strikes": "strikes"}, inplace=True)

# We no longer need to infer pitcher hand from "Relside" because "Pitcherhand" is provided.
# (Similarly, Batterside is provided as "batterHand" renamed to "Batterside".)

############################################
# Pitch Category Grouping
############################################
pitch_categories = {
    "Breaking Ball": ["Slider", "Curveball"],
    "Fastball": ["Fastball", "Four-Seam", "Sinker", "Cutter"],
    "Offspeed": ["ChangeUp", "Splitter"]
}

def categorize_pitch_type(pitch_type):
    for category, pitches in pitch_categories.items():
        if pitch_type in pitches:
            return category
    return "Other"

df['Pitchcategory'] = df['Autopitchtype'].apply(categorize_pitch_type)

############################################
# Create Boolean Count Category Columns
############################################
df['Firstpitch'] = (df['balls'] == 0) & (df['strikes'] == 0)
df['Twostrike'] = df['strikes'] == 2
df['Threeball'] = df['balls'] == 3
df['Evencount'] = (df['balls'] == df['strikes']) & (df['balls'] != 0)
df['Hitterfriendly'] = df['balls'] > df['strikes']
df['Pitcherfriendly'] = df['strikes'] > df['balls']

############################################
# Ensure Numeric Columns Are Correctly Typed
############################################
df['Exitspeed'] = pd.to_numeric(df['Exitspeed'], errors='coerce')
df['Angle'] = pd.to_numeric(df['Angle'], errors='coerce')

############################################
# Convert Date Column and Filter by Date
############################################
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
min_date = df['Date'].min()
max_date = df['Date'].max()
start_date, end_date = st.sidebar.date_input("Select Date Range", value=[min_date, max_date])
start_dt = pd.to_datetime(start_date)
end_dt = pd.to_datetime(end_date)
df = df[(df['Date'] >= start_dt) & (df['Date'] <= end_dt)]

############################################
# Streamlit Sidebar Filters
############################################
st.sidebar.header("Filter Options")

# Batter Filter
batters = df['Batter'].dropna().unique()
batters = sorted(batters)
batters = ["All Hitters"] + list(batters)
default_batter = ["All Hitters"] if "All Hitters" in batters else [batters[0]] if batters else []
selected_batters = st.sidebar.multiselect("Select Batter(s)", batters, default=default_batter)
batter_filter = True if "All Hitters" in selected_batters else df['Batter'].isin(selected_batters)

# Pitcher Hand Filter (using provided Pitcherhand)
pitcher_hands = ['All', 'R', 'L']
selected_pitcher_hand = st.sidebar.selectbox("Pitcher Hand", pitcher_hands, index=0)

# Higher-level Pitch Categories Filter
pitch_categories_list = list(pitch_categories.keys())
if 'Other' not in pitch_categories_list:
    pitch_categories_list.append('Other')
selected_categories = st.sidebar.multiselect("Select Pitch Category(s)", pitch_categories_list, default=pitch_categories_list)

# Specific Pitch Types Filter – populate from the selected categories.
available_pitch_types = []
for category in selected_categories:
    if category in pitch_categories:
        available_pitch_types.extend(pitch_categories[category])
    else:
        # For 'Other', select pitch types not in any known category.
        categorized = [p for pitches in pitch_categories.values() for p in pitches]
        other_types = df[~df['Autopitchtype'].isin(categorized)]['Autopitchtype'].dropna()
        available_pitch_types.extend(other_types.astype(str).str.strip().tolist())

available_pitch_types = sorted(list(set([pt for pt in available_pitch_types if pt and pt.lower() != 'nan'])))
selected_pitch_types = st.sidebar.multiselect("Select Pitch Type(s)", available_pitch_types, default=available_pitch_types)

# Map count options for later use (if needed)
count_option_to_column = {
    '1st-pitch': 'Firstpitch',
    '2-Strike': 'Twostrike',
    '3-Ball': 'Threeball',
    'Even': 'Evencount',
    'Hitter-Friendly': 'Hitterfriendly',
    'Pitcher-Friendly': 'Pitcherfriendly'
}

# Create Event Column (combining Pitchcall and Playresult)
df['Event'] = np.where(
    df['Playresult'].isin(['Undefined', 'StolenBase', 'CaughtStealing']),
    np.where(
        df['Pitchcall'].eq('BallInDirt'),
        'BallCalled',
        np.where(
            df['Pitchcall'].str.contains('Foul', case=False, na=False),
            'Foul',
            df['Pitchcall']
        )
    ),
    df['Playresult']
)

############################################
# Define Swing vs Take based on Exitspeed/Pitchcall
############################################
df['Swing'] = np.where(
    (df['Exitspeed'] > 0) | (df['Pitchcall'].str.contains('Swing|Foul', case=False, na=False)),
    'Swing',    # Swing if there is a nonzero Exitspeed or "Swing"/"Foul" in call.
    'Take'
)

############################################
# Define Strike Zone and Zone Column
############################################
STRIKE_ZONE_SIDE = (-0.83, 0.83)
STRIKE_ZONE_HEIGHT = (1.5, 3.5)
df['Zone'] = np.where(
    (df['Platelocside'].between(*STRIKE_ZONE_SIDE)) &
    (df['Platelocheight'].between(*STRIKE_ZONE_HEIGHT)),
    'InZone',
    'Out'
)

############################################
# Create Unique At-Bat ID Using Date, Pitcher, Paofinning, Inning
############################################
df['Atbatid'] = (
    df['Date'].astype(str).fillna('') + '_' +
    df['Pitcher'].fillna('').astype(str) + '_' +
    df['Paofinning'].fillna('').astype(str) + '_' +
    df['Inning'].fillna('').astype(str)
)

############################################
# Contact and Whiff Columns Based on Exitspeed
############################################
df['Contact'] = np.where(df['Exitspeed'] > 0, 'Yes', 'No')
df['Whiff'] = np.where((df['Swing'] == 'Swing') & (df['Contact'] == 'No'), 'Yes', 'No')

############################################
# Create Final Count String (if needed)
############################################
df['Count'] = df['balls'].astype(str) + '-' + df['strikes'].astype(str)

df['ContactPct'] = np.where(
    df['Swing'] == 'Swing',
    np.where(df['Contact'] == 'Yes', 1.0, 0.0),
    np.nan
)

############################################
# Map Plate Zones (Heart, Shadow, Chase, Waste) Based on Plate Location
############################################
def map_plate_zone(row):
    side = row['Platelocside']
    height = row['Platelocheight']
    HEART_SIDE = (-0.56, 0.56)
    HEART_HEIGHT = (1.83, 3.17)
    SHADOW_SIDE = (-1.11, 1.11)
    SHADOW_HEIGHT = (1.17, 3.83)
    CHASE_SIDE = (-1.67, 1.67)
    CHASE_HEIGHT = (0.5, 4.33)
    if HEART_SIDE[0] <= side <= HEART_SIDE[1] and HEART_HEIGHT[0] <= height <= HEART_HEIGHT[1]:
        return 'Heart'
    elif SHADOW_SIDE[0] <= side <= SHADOW_SIDE[1] and SHADOW_HEIGHT[0] <= height <= SHADOW_HEIGHT[1]:
        return 'Shadow'
    elif CHASE_SIDE[0] <= side <= CHASE_SIDE[1] and CHASE_HEIGHT[0] <= height <= CHASE_HEIGHT[1]:
        return 'Chase'
    else:
        return 'Waste'

df['PlateZone'] = df.apply(map_plate_zone, axis=1)

############################################
# Split Data into Swing and Take for Model Predictions
############################################
df_no_swing = df[df['Swing'] == 'Take'].dropna(subset=['Platelocside', 'Platelocheight', 'strikes', 'balls']).copy()
df_swing = df[df['Swing'] == 'Swing'].dropna(subset=['Platelocside', 'Platelocheight', 'strikes', 'balls']).copy()

df_no_swing['decision_rv'] = model_no_swing.predict(df_no_swing[['Platelocside','Platelocheight','strikes','balls']])
df_swing['decision_rv'] = model_swing.predict(df_swing[['Platelocside','Platelocheight','strikes','balls']])

# Merge predictions back using "Pitchuid" (from uniqPitchId)
if 'Pitchuid' in df.columns:
    df_no_swing_merge = df_no_swing[['Pitchuid','decision_rv']].rename(columns={'decision_rv': 'decision_rv_no_swing'})
    df_swing_merge = df_swing[['Pitchuid','decision_rv']].rename(columns={'decision_rv': 'decision_rv_swing'})
    df = df.merge(df_no_swing_merge, on='Pitchuid', how='left').merge(df_swing_merge, on='Pitchuid', how='left')
    df['decision_rv'] = df['decision_rv_no_swing'].combine_first(df['decision_rv_swing'])
else:
    pass

# ############################################
# # Adjust Pitcher Types (Example: assign “Machine” or “Scrimmage”)
# ############################################
# df['Pitcher'] = df['Pitcher'].fillna('Machine')
# df['Pitchertype'] = df['Pitcher'].apply(lambda x: 'Machine' if x == 'Bunnell, Jack' else 'Scrimmage')
# pitchers = sorted(df['Pitchertype'].unique())
# selected_pitchers = st.sidebar.multiselect("Select Pitcher(s)", pitchers, default=pitchers)
# pitcher_filter = df['Pitchertype'].isin(selected_pitchers)

# ############################################
# # Apply Overall Filters to Create Final Data Subsets
# ############################################
# all_pitches = df[
#     batter_filter &
#     pitcher_filter &
#     df['Pitchcategory'].isin(selected_categories) &
#     df['Autopitchtype'].isin(selected_pitch_types)
# ]
# if selected_pitcher_hand != 'All':
#     all_pitches = all_pitches[all_pitches['Pitcherhand'] == selected_pitcher_hand]

# filtered_data = all_pitches[(all_pitches['Exitspeed'] > 0) & (all_pitches['Exitspeed'].notnull())]
# if selected_pitcher_hand != 'All':
#     filtered_data = filtered_data[filtered_data['Pitcherhand'] == selected_pitcher_hand]

# ############################################
# # xSLG Prediction: Create Interaction & Predict
# ############################################
# filtered_data['interaction'] = filtered_data['Exitspeed'] * filtered_data['Angle']
# X_pred = filtered_data[['Exitspeed', 'Angle', 'interaction']].copy()
# X_pred.rename(columns={'Exitspeed':'launch_speed', 'Angle':'launch_angle'}, inplace=True)
# filtered_data['xSLG'] = best_model.predict(X_pred)
# if 'Pitchuid' in filtered_data.columns and 'Pitchuid' in all_pitches.columns:
#     xSLG_data = filtered_data[['Pitchuid', 'xSLG']].drop_duplicates(subset='Pitchuid')
#     all_pitches = all_pitches.merge(xSLG_data, on='Pitchuid', how='left')
#     all_pitches['xSLG'] = all_pitches['xSLG'].fillna(np.nan)

# ############################################
# # Visualization Functions
# ############################################
# def create_heatmap(data, metric, ax):
#     if data.empty or metric not in data.columns:
#         ax.set_title(f"No data available for {metric}.")
#         ax.axis('off')
#         return
#     x_min, x_max = -2.5, 2.5
#     y_min, y_max = 0, 5
#     x_bins = np.linspace(x_min, x_max, 10)
#     y_bins = np.linspace(y_min, y_max, 10)
#     heatmap_data, _, _ = np.histogram2d(
#         data['Platelocside'], data['Platelocheight'],
#         bins=[x_bins, y_bins],
#         weights=data[metric],
#         density=False
#     )
#     counts, _, _ = np.histogram2d(
#         data['Platelocside'], data['Platelocheight'],
#         bins=[x_bins, y_bins]
#     )
#     with np.errstate(divide='ignore', invalid='ignore'):
#         heatmap_data = np.divide(heatmap_data, counts, out=np.full_like(heatmap_data, np.nan), where=counts != 0)
#     heatmap_data = np.ma.masked_invalid(heatmap_data)
#     if metric == 'xSLG' and np.isnan(heatmap_data).all():
#         ax.set_title("No data available for xSLG.")
#         ax.axis('off')
#         return
#     if metric == 'Exitspeed':
#         vmin, vmax = 60, 100
#     elif metric == 'Angle':
#         vmin, vmax = -20, 40
#     elif metric == 'xSLG':
#         vmin, vmax = 0.25, 0.65
#     elif metric == 'decision_rv':
#         vmin, vmax = -0.2, 0.15
#     elif metric == 'ContactPct':
#         vmin, vmax = 0.7, 1.0
#     else:
#         vmin, vmax = np.nanmin(heatmap_data), np.nanmax(heatmap_data)
#     extent = [x_min, x_max, y_min, y_max]
#     im = ax.imshow(heatmap_data.T, cmap='coolwarm', origin='lower', extent=extent, aspect='auto', vmin=vmin, vmax=vmax)
#     cbar = plt.colorbar(im, ax=ax)
#     cbar.set_label(metric)
#     ax.add_patch(plt.Rectangle((-0.83, 1.5), 1.66, 2.1, edgecolor='black', facecolor='none', lw=2))
#     plate_vertices = [(-0.83, 0.1), (0.83, 0.1), (0.65, 0.25), (0, 0.5), (-0.65, 0.25)]
#     plate = plt.Polygon(plate_vertices, closed=True, linewidth=1, edgecolor='k', facecolor='none')
#     ax.add_patch(plate)
#     ax.set_title(metric, fontsize=20)
#     ax.set_xlabel('PlateLocSide')
#     ax.set_ylabel('PlateLocHeight')

# def plot_pitch_locations_by_playresult(data):
#     if data.empty:
#         st.warning("No data available for the selected filters to plot pitch locations.")
#         return
#     data['Swing'] = data['Swing'].astype('category')
#     data['xSLG'] = pd.to_numeric(data['xSLG'].fillna(0), errors='coerce').fillna(0)
#     data['Count'] = data['balls'].astype(str) + '-' + data['strikes'].astype(str)
#     pitcher_sides = ['R', 'L']
#     swing_types = ['Swing', 'Take']
#     plate_vertices = [(-0.83, 0.1), (0.83, 0.1), (0.65, 0.25), (0, 0.5), (-0.65, 0.25)]
#     fig, axes = plt.subplots(2, 2, figsize=(14, 16), sharey=True, sharex=True, gridspec_kw={'height_ratios': [1, 1]})
#     axes = axes.flatten()
#     norm = plt.Normalize(vmin=0, vmax=0.8)
#     sm = plt.cm.ScalarMappable(cmap='coolwarm', norm=norm)
#     sm.set_array([])
#     zone_definitions = {
#         'Heart': {'x_range': (-0.56, 0.56), 'y_range': (1.83, 3.17), 'color': '#7CFC00', 'alpha': 0.3},
#         'Shadow': {'x_range': (-1.11, 1.11), 'y_range': (1.17, 3.83), 'color': '#FFD700', 'alpha': 0.2},
#         'Chase': {'x_range': (-1.67, 1.67), 'y_range': (0.5, 4.33), 'color': '#FFA07A', 'alpha': 0.15},
#         'Waste': {'x_range': (-2.5, 2.5), 'y_range': (0.0, 5.0), 'color': '#FF6347', 'alpha': 0.1}
#     }
#     for i, (swing, pitcher_side) in enumerate([(s, p) for s in swing_types for p in pitcher_sides]):
#         side_data = data[(data['Swing'] == swing) & (data['Pitcherhand'] == pitcher_side)]
#         for _, row in side_data.iterrows():
#             marker = 'o' if row['Whiff'] == 'No' else 'x'
#             axes[i].scatter(row['Platelocside'], row['Platelocheight'], c=row['xSLG'], cmap='coolwarm',
#                             norm=norm, edgecolor='black', s=100, marker=marker, zorder=3)
#             if swing == 'Take':
#                 axes[i].text(row['Platelocside'] - 0.15, row['Platelocheight'], row['Count'],
#                              fontsize=8, color='darkblue', ha='right', va='center', zorder=4)
#         for zone, props in zone_definitions.items():
#             axes[i].add_patch(Rectangle((props['x_range'][0], props['y_range'][0]),
#                                           props['x_range'][1] - props['x_range'][0],
#                                           props['y_range'][1] - props['y_range'][0],
#                                           edgecolor='none', facecolor=props['color'], alpha=props['alpha'], zorder=0))
#         axes[i].add_patch(Rectangle((-0.83, 1.5), 1.66, 2.1, edgecolor='black', facecolor='none', lw=2))
#         plate = Polygon(plate_vertices, closed=True, linewidth=1, edgecolor='k', facecolor='none')
#         axes[i].add_patch(plate)
#         axes[i].set_title(f'{swing} vs {pitcher_side}-Handed Pitchers')
#         axes[i].set_xlim(-2.5, 2.5)
#         axes[i].set_ylim(0, 5)
#         axes[i].set_xlabel('PlateLocSide')
#         axes[i].set_ylabel('PlateLocHeight')
#         if axes[i].get_legend() is not None:
#             axes[i].get_legend().remove()
#     cbar_ax = fig.add_axes([0.3, 0.5, 0.4, 0.02])
#     cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
#     cbar.set_label('xSLG (0 - 0.8)', fontsize=12, color='white')
#     cbar.ax.tick_params(labelcolor='white')
#     fig.text(cbar_ax.get_position().x0 - 0.15, cbar_ax.get_position().y0, 'X = WHIFF', ha='right', va='center', fontsize=20, color='white')
#     plt.subplots_adjust(hspace=0.4)
#     st.pyplot(fig)

# def plot_pitch_locations_by_hand_and_ypred(data):
#     import matplotlib.pyplot as plt
#     from matplotlib.patches import Rectangle, Polygon
#     if data.empty:
#         st.warning("No data available for the selected filters.")
#         return
#     shape_map = {
#         'BallCalled': 'o',
#         'Foul': '^',
#         'StrikeCalled': 's',
#         'StrikeSwinging': 'D',
#         'HitByPitch': 'p',
#         'Single': 'v',
#         'Double': '>',
#         'Triple': '<',
#         'HomeRun': '*',
#         'Out': 'X',
#         'Undefined': '8'
#     }
#     zone_definitions = {
#         'Heart':  {'x_range': (-0.56, 0.56), 'y_range': (1.83, 3.17), 'color': '#7CFC00', 'alpha': 0.3},
#         'Shadow': {'x_range': (-1.11, 1.11), 'y_range': (1.17, 3.83), 'color': '#FFD700', 'alpha': 0.2},
#         'Chase':  {'x_range': (-1.67, 1.67), 'y_range': (0.5, 4.33),  'color': '#FFA07A', 'alpha': 0.15},
#         'Waste':  {'x_range': (-2.5, 2.5),   'y_range': (0.0, 5.0),   'color': '#FF6347', 'alpha': 0.1}
#     }
#     pitcher_sides = ['R', 'L']
#     fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)
#     for i, side in enumerate(pitcher_sides):
#         subset = data[data['Pitcherhand'] == side]
#         for event_type in subset['Event'].unique():
#             sub_e = subset[subset['Event'] == event_type]
#             marker_style = shape_map.get(event_type, 'o')
#             sc = axes[i].scatter(sub_e['Platelocside'], sub_e['Platelocheight'],
#                                  c=sub_e['decision_rv'], cmap='coolwarm', marker=marker_style,
#                                  edgecolor='black', vmin=-0.2, vmax=0.15, s=80, alpha=0.7)
#         for zone, props in zone_definitions.items():
#             axes[i].add_patch(Rectangle((props['x_range'][0], props['y_range'][0]),
#                                           props['x_range'][1]-props['x_range'][0],
#                                           props['y_range'][1]-props['y_range'][0],
#                                           edgecolor='none', facecolor=props['color'],
#                                           alpha=props['alpha'], zorder=0))
#         axes[i].add_patch(Rectangle((-0.83,1.5),1.66,2.0, edgecolor='black', facecolor='none', lw=2))
#         axes[i].set_title(f"Pitcher Side: {side}")
#         axes[i].set_xlim(-2.5,2.5)
#         axes[i].set_ylim(0,5)
#         axes[i].set_xlabel("PlateLocSide")
#         axes[i].set_ylabel("PlateLocHeight")
#     cbar = fig.colorbar(sc, ax=axes.ravel().tolist())
#     cbar.set_label("decision_rv", fontsize=12)
#     handles = []
#     for event_name, marker_shape in shape_map.items():
#         handle = mlines.Line2D([], [], color='darkblue', marker=marker_shape,
#                                markersize=8, label=event_name, linestyle='None')
#         handles.append(handle)
#     legend_obj = fig.legend(handles=handles, loc='right', frameon=True, title='Shapes')
#     legend_obj.get_title().set_color('darkblue')
#     for text_obj in legend_obj.get_texts():
#         text_obj.set_color('darkblue')
#     st.pyplot(fig)

# def plot_kde_comparison(data):
#     f_league_path = "league_kde_earliest.npy"
#     x_grid_path = "grid_x.npy"
#     y_grid_path = "grid_y.npy"
#     try:
#         f_league = np.load(f_league_path, allow_pickle=True)
#         X = np.load(x_grid_path, allow_pickle=True)
#         Y = np.load(y_grid_path, allow_pickle=True)
#     except FileNotFoundError as e:
#         st.error(f"File not found: {e}")
#         return
#     except Exception as e:
#         st.error(f"Error loading .npy files: {e}")
#         return
#     if data.empty or 'Direction' not in data.columns or 'Angle' not in data.columns:
#         st.error("The dataset is missing required columns ('Direction', 'Angle').")
#         return
#     try:
#         x_loc_player = data['Direction']
#         y_loc_player = data['Angle']
#         values_player = np.vstack([x_loc_player, y_loc_player])
#         kernel_player = gaussian_kde(values_player)
#         f_player = np.reshape(kernel_player(np.vstack([X.ravel(), Y.ravel()])).T, X.shape)
#         f_player = f_player * (100 / f_player.sum())
#     except Exception as e:
#         st.error(f"Error in KDE computation: {e}")
#         return
#     try:
#         kde_difference = f_player - f_league
#         fig, ax = plt.subplots(figsize=(7,7))
#         levels = list(range(-13,14,2))
#         cfset = ax.contourf(X, Y, kde_difference * 1000, levels=levels, cmap='vlag', extend='both')
#         ax.set_xlim(0,90)
#         ax.set_ylim(-30,60)
#         ax.set_xticks([])
#         ax.set_yticks([])
#         x_ticks = [0,30,60,90]
#         x_labels = ['Pull','Center','Oppo']
#         for label, pos0, pos1 in zip(x_labels, x_ticks[:-1], x_ticks[1:]):
#             ax.text((pos0+pos1)/2, -35, label, ha='center', va='top', fontsize=12)
#         y_ticks = [-30,10,25,50,60]
#         y_labels = ['Ground Ball','Line Drive','Fly Ball','Pop Up']
#         for label, pos0, pos1 in zip(y_labels, y_ticks[:-1], y_ticks[1:]):
#             ax.text(-10, (pos0+pos1)/2, label, ha='right', va='center', fontsize=12)
#         sm = plt.cm.ScalarMappable(cmap='vlag', norm=mpl.colors.BoundaryNorm(levels, ncolors=256))
#         sm.set_array([])
#         cbar = fig.colorbar(sm, ax=ax, orientation='vertical', shrink=0.8, pad=0.05)
#         cbar.ax.axis('off')
#         for label, position, color in zip(['Less\nOften','Same','More\nOften'], [-24,15,53.5],
#                                            [sns.color_palette('vlag',25)[0],'k',sns.color_palette('vlag',25)[-1]]):
#             cbar.ax.text(1.15, position, label, ha='center', va='center', color=color, fontsize=12, transform=ax.get_yaxis_transform())
#         ax.set_title("Batted Ball Difference", fontsize=16)
#         fig.text(0.5, 0.01, "batted-ball-charts.streamlit.app | Data: Baseball Savant/pybaseball", ha='center', fontsize=8)
#         sns.despine()
#         st.pyplot(fig)
#     except Exception as e:
#         st.error(f"Error in plotting: {e}")

# def create_spray_chart(data, ax):
#     outline_points = [
#         (0, 0),
#         (-45, 90),
#         (-45, 315),
#         (-15, 375),
#         (0, 405),
#         (15, 375),
#         (45, 325),
#         (45, 90),
#         (0, 128),
#         (-45, 90),
#         (0, 0)
#     ]
#     outline_cartesian = [(-distance * np.sin(np.radians(angle)), distance * np.cos(np.radians(angle)))
#                          for angle, distance in outline_points]
#     outline_x, outline_y = zip(*outline_cartesian)
#     ax.plot(outline_x, outline_y, color='red', linewidth=2, label="Field Outline")
#     if 'Direction' in data.columns and 'Distance' in data.columns:
#         data['Direction_rad'] = np.radians(data['Direction'])
#         data['Rotated_X'] = -data['Distance'] * np.sin(data['Direction_rad'])
#         data['Rotated_Y'] = data['Distance'] * np.cos(data['Direction_rad'])
#         scatter = ax.scatter(data['Rotated_X'], data['Rotated_Y'], c=data['Exitspeed'], cmap='coolwarm', s=50, 
#                              edgecolor='k', label='Hits', vmin=60, vmax=100)
#         cbar = plt.colorbar(scatter, ax=ax)
#         cbar.set_label("Exit Speed")
#     ax.set_title("Spray Chart")
#     ax.set_xlabel("X (Feet)")
#     ax.set_ylabel("Y (Feet)")
#     ax.set_xlim([-250, 250])
#     ax.set_ylim([0,450])
#     ax.set_aspect('equal')
#     ax.legend()

# def display_hitter_metrics(all_pitches):
#     if all_pitches.empty:
#         st.write("No data available for the selected filters.")
#         return
#     grouped = all_pitches.groupby('Batter')
#     rows = []
#     for batter, group_data in grouped:
#         total_events = len(group_data)
#         avg_ev = group_data['Exitspeed'].mean() if 'Exitspeed' in group_data else np.nan
#         max_ev = group_data['Exitspeed'].max() if 'Exitspeed' in group_data else np.nan
#         avg_launch_angle = group_data['Angle'].mean() if 'Angle' in group_data else np.nan
#         hard_hit_count = (group_data['Exitspeed'] > 90).sum()
#         hard_hit_pct = hard_hit_count / total_events if total_events > 0 else np.nan
#         barrel_mask = (group_data['Exitspeed'] >= 99) & (group_data['Angle'].between(25, 31))
#         barrel_count = barrel_mask.sum()
#         barrel_pct = barrel_count / total_events if total_events > 0 else np.nan
#         gb_count = (group_data['Angle'] < 0).sum()
#         gb_pct = gb_count / total_events if total_events > 0 else np.nan
#         if 'Batterside' in group_data.columns and not group_data['Batterside'].isnull().all():
#             batter_sides = group_data['Batterside'].dropna().unique()
#             batter_side = batter_sides[0] if len(batter_sides) > 0 else None
#         else:
#             batter_side = None
#         if batter_side == 'Left':
#             pull_count = (group_data['Direction'] > 0).sum()
#         elif batter_side == 'Right':
#             pull_count = (group_data['Direction'] < 0).sum()
#         else:
#             pull_count = np.nan
#         pull_pct = pull_count / total_events if total_events > 0 and not np.isnan(pull_count) else np.nan
#         pop_fly_count = ((group_data['Angle'] > 50) & (group_data['Exitspeed'] < 85)).sum()
#         pop_fly_pct = pop_fly_count / total_events if total_events > 0 else np.nan
#         avg_xSLG = group_data.loc[group_data['Swing'] == 'Swing', 'xSLG'].mean() if 'xSLG' in group_data else np.nan
#         o_swing = ((group_data['Zone'] == 'Out') & (group_data['Swing'] == 'Swing')).sum()
#         z_swing = ((group_data['Zone'] == 'InZone') & (group_data['Swing'] == 'Swing')).sum()
#         total_swings = (group_data['Swing'] == 'Swing').sum()
#         total_pitches = len(group_data)
#         o_contact = ((group_data['Zone'] == 'Out') & (group_data['Swing'] == 'Swing') & (group_data['Contact'] == 'Yes')).sum()
#         z_contact = ((group_data['Zone'] == 'InZone') & (group_data['Swing'] == 'Swing') & (group_data['Contact'] == 'Yes')).sum()
#         total_contact = (group_data['Contact'] == 'Yes').sum()
#         zone_pitches = (group_data['Zone'] == 'InZone').sum()
#         first_strike = ((group_data['Pitchcall'] == 'Strike') & (group_data['Swing'] == 'Take')).sum()
#         swinging_strike = ((group_data['Swing'] == 'Swing') & (group_data['Contact'] == 'No')).sum()
#         o_swing_pct = o_swing / np.maximum((group_data['Zone'] == 'Out').sum(), 1)
#         z_swing_pct = z_swing / np.maximum((group_data['Zone'] == 'InZone').sum(), 1)
#         swing_pct = total_swings / np.maximum(total_pitches, 1)
#         o_contact_pct = o_contact / np.maximum(o_swing, 1)
#         z_contact_pct = z_contact / np.maximum(z_swing, 1)
#         contact_pct = total_contact / np.maximum(total_swings, 1)
#         zone_pct = zone_pitches / np.maximum(total_pitches, 1)
#         f_strike_pct = first_strike / np.maximum(group_data['Atbatid'].nunique(), 1)
#         swstr_pct = swinging_strike / np.maximum(total_pitches, 1)
#         def fmt_num(val, decimals=2):
#             return f"{val:.{decimals}f}" if pd.notna(val) else np.nan
#         def fmt_pct(val):
#             return f"{val*100:.2f}%" if pd.notna(val) else np.nan
#         def fmt_xslg(val):
#             return f"{val:.3f}" if pd.notna(val) else np.nan
#         rows.append({
#             'Batter': batter,
#             'Avg EV': fmt_num(avg_ev),
#             'Max EV': fmt_num(max_ev),
#             'Avg LA': fmt_num(avg_launch_angle),
#             'xSLG': fmt_xslg(avg_xSLG),
#             'Hard Hit%': fmt_pct(hard_hit_pct),
#             'Barrel%': fmt_pct(barrel_pct),
#             'O-Swing%': fmt_pct(o_swing_pct),
#             'Z-Swing%': fmt_pct(z_swing_pct),
#             'Swing%': fmt_pct(swing_pct),
#             'O-Contact%': fmt_pct(o_contact_pct),
#             'Z-Contact%': fmt_pct(z_contact_pct),
#             'Contact%': fmt_pct(contact_pct),
#             'Zone%': fmt_pct(zone_pct),
#             'F-Strike%': fmt_pct(f_strike_pct),
#             'SwStr%': fmt_pct(swstr_pct),
#             'GB%': fmt_pct(gb_pct),
#             'PULL%': fmt_pct(pull_pct),
#             'POP FLY%': fmt_pct(pop_fly_pct)
#         })
#     metrics_df = pd.DataFrame(rows)
#     def apply_20_80_scale(mean_pred, mu, std):
#         if pd.isna(mean_pred):
#             return np.nan
#         if std == 0:
#             return 50
#         z = (mean_pred - mu) / std
#         return np.clip(50 + 10*z, 20, 80)
#     df_no_swing_sub = all_pitches[all_pitches['Swing'] == 'Take']
#     df_no_swing_group = df_no_swing_sub.groupby('Batter').agg(mean_pred_no_swing=('decision_rv_no_swing','mean'),
#                                                               pitches_no_swing=('decision_rv_no_swing','count')).reset_index()
#     df_swing_sub = all_pitches[all_pitches['Swing'] == 'Swing']
#     df_swing_group = df_swing_sub.groupby('Batter').agg(mean_pred_swing=('decision_rv_swing','mean'),
#                                                         pitches_swing=('decision_rv_swing','count')).reset_index()
#     df_overall_group = all_pitches.groupby('Batter').agg(mean_pred_overall=('decision_rv','mean'),
#                                                          pitches_overall=('decision_rv','count')).reset_index()
#     df_dv_merged = pd.merge(df_no_swing_group, df_swing_group, on='Batter', how='outer')
#     df_dv_merged = pd.merge(df_dv_merged, df_overall_group, on='Batter', how='outer')
#     mu_no_swing  = 0.0119
#     std_no_swing = 0.0199
#     mu_swing     = -0.0194
#     std_swing    = 0.0129
#     mu_overall   = -0.0032
#     std_overall  = 0.0130
#     df_dv_merged['decision_value_no_swing'] = df_dv_merged['mean_pred_no_swing'].apply(lambda x: apply_20_80_scale(x, mu_no_swing, std_no_swing))
#     df_dv_merged['decision_value_swing'] = df_dv_merged['mean_pred_swing'].apply(lambda x: apply_20_80_scale(x, mu_swing, std_swing))
#     df_dv_merged['decision_value_overall'] = df_dv_merged['mean_pred_overall'].apply(lambda x: apply_20_80_scale(x, mu_overall, std_overall))
#     df_dv_merged['decision_value_take'] = df_dv_merged['decision_value_no_swing'].round(1)
#     df_dv_merged['decision_value_swing'] = df_dv_merged['decision_value_swing'].round(1)
#     df_dv_merged['decision_value_overall'] = df_dv_merged['decision_value_overall'].round(1)
#     final_df = metrics_df.merge(df_dv_merged[['Batter','pitches_no_swing','pitches_swing','pitches_overall',
#                                               'decision_value_take','decision_value_swing','decision_value_overall']],
#                                 on='Batter', how='right')
#     st.write("### Hitter Metrics + Decision Values (20–80)")
#     st.dataframe(final_df.fillna('N/A'))

# def calculate_zone_metrics(data):
#     if 'PlateZone' not in data.columns:
#         st.error("The column 'PlateZone' does not exist in the dataset.")
#         return
#     zones = ['Heart', 'Shadow', 'Chase', 'Waste']
#     zone_metrics = []
#     for zone in zones:
#         zone_data = data[data['PlateZone'] == zone]
#         zone_swings = zone_data[zone_data['Swing'] == 'Swing']
#         total_pitches = len(zone_data)
#         swings = len(zone_swings)
#         contacts = (zone_swings['Contact'] == 'Yes').sum()
#         hard_hits = (zone_swings['Exitspeed'] > 90).sum() if 'Exitspeed' in zone_swings.columns else 0
#         xslg = zone_swings['xSLG'].mean() if not zone_swings['xSLG'].isnull().all() else np.nan
#         swing_pct = swings / total_pitches if total_pitches > 0 else 0
#         contact_pct = contacts / swings if swings > 0 else 0
#         hard_hit_pct = hard_hits / swings if swings > 0 else 0
#         zone_metrics.append({
#             'Zone': zone,
#             'Total Pitches': total_pitches,
#             'Swing%': round(swing_pct,4),
#             'Contact%': round(contact_pct,4),
#             'xSLG': round(xslg,4) if pd.notnull(xslg) else 'N/A',
#             'Hard Hit%': round(hard_hit_pct,4)
#         })
#     zone_metrics_df = pd.DataFrame(zone_metrics)
#     narrow_style = """
#     <style>
#     table td, table th {
#         width: 60px !important;
#         text-align: center !important;
#     }
#     </style>
#     """
#     st.write("### Zone Metrics Overview")
#     st.markdown(narrow_style, unsafe_allow_html=True)
#     st.dataframe(zone_metrics_df)

# ############################################
# # Streamlit Navigation
# ############################################
# st.sidebar.title("Navigation")
# page = st.sidebar.radio("Select Page", ["Heatmaps", "Pitch Locations by Playresult", "Pitch Locations by Decision Value", "Batted Ball Outcomes", "Spray Chart", "Hitter Metrics", "Zone Metrics", "Raw Data"])

# if page == "Heatmaps":
#     st.title("Hitter Heatmaps")
#     fig, axs = plt.subplots(3, 2, figsize=(18, 28))
#     if 'Angle' in filtered_data.columns and not filtered_data['Angle'].isnull().all():
#         create_heatmap(filtered_data, 'Angle', axs[0,0])
#     else:
#         axs[0,0].set_title("Launch Angle")
#         axs[0,0].axis('off')
#         axs[0,0].text(0.5,0.5,"Launch Angle Heatmap\n(Data Not Available)", horizontalalignment='center', verticalalignment='center')
#     if 'Exitspeed' in filtered_data.columns and not filtered_data['Exitspeed'].isnull().all():
#         create_heatmap(filtered_data, 'Exitspeed', axs[0,1])
#     else:
#         axs[0,1].set_title("Exit Velocity")
#         axs[0,1].axis('off')
#         axs[0,1].text(0.5,0.5,"Exit Velocity Heatmap\n(Data Not Available)", horizontalalignment='center', verticalalignment='center')
#     if 'xSLG' in filtered_data.columns and not filtered_data['xSLG'].isnull().all():
#         create_heatmap(filtered_data, 'xSLG', axs[1,0])
#     else:
#         axs[1,0].set_title("xSLG")
#         axs[1,0].axis('off')
#         axs[1,0].text(0.5,0.5,"xSLG Heatmap\n(Data Not Available)", horizontalalignment='center', verticalalignment='center')
#     if 'decision_rv' in all_pitches.columns and not all_pitches['decision_rv'].isnull().all():
#         create_heatmap(all_pitches, 'decision_rv', axs[1,1])
#     else:
#         axs[1,1].set_title("Decision Value")
#         axs[1,1].axis('off')
#         axs[1,1].text(0.5,0.5,"Decision Value Heatmap\n(Data Not Available)", horizontalalignment='center', verticalalignment='center')
#     swing_data = all_pitches[(all_pitches['Swing'] == 'Swing') & (all_pitches['Platelocside'].notnull()) & (all_pitches['Platelocheight'].notnull())]
#     if 'ContactPct' in swing_data.columns and not swing_data['ContactPct'].isnull().all():
#         create_heatmap(swing_data, 'ContactPct', axs[2,0])
#     else:
#         axs[2,0].set_title("Contact%")
#         axs[2,0].axis('off')
#         axs[2,0].text(0.5,0.5,"Contact% Heatmap\n(Data Not Available)", ha='center', va='center')
#     plt.tight_layout()
#     st.pyplot(fig)
# elif page == "Spray Chart":
#     st.title("Spray Chart")
#     spray_data = filtered_data[(filtered_data['Direction'].notnull()) & (filtered_data['Distance'].notnull())]
#     fig, ax = plt.subplots(figsize=(10,8))
#     create_spray_chart(spray_data, ax)
#     st.pyplot(fig)
# elif page == "Pitch Locations by Playresult":
#     st.title("Pitch Locations by Playresult")
#     plot_pitch_locations_by_playresult(all_pitches)
# elif page == "Pitch Locations by Decision Value":
#     st.title("Pitch Locations by Decision Value")
#     plot_pitch_locations_by_hand_and_ypred(all_pitches)
# elif page == "Raw Data":
#     st.title("Raw Data Display")
#     st.write("### Raw Data")
#     if all_pitches.empty:
#         st.warning("No filtered data available. Adjust your filters to see results.")
#     else:
#         st.dataframe(all_pitches.head(1000))
# elif page == "Hitter Metrics":
#     st.title("Hitter Metrics")
#     display_hitter_metrics(all_pitches)
# elif page == "Batted Ball Outcomes":
#     st.title("Batted Ball Outcomes")
#     if filtered_data.empty:
#         st.warning("No data available for the selected filters.")
#     else:
#         plot_kde_comparison(filtered_data)
# elif page == "Zone Metrics":
#     st.title("Zone Metrics: Heart, Shadow, Chase, Waste")
#     calculate_zone_metrics(all_pitches)


In [ ]:
df.head()